# 实验 2：解剖 Qwen3-0.6B 的一次完整前向传播

## 今天回答的问题

**Qwen3-0.6B 对一段真实 token 序列究竟进行了哪些计算，这些计算能否被我们自己重新实现并验证？**

实验 1 画出了模型的外部骨架，但没有任何数据真的流过去。本实验补上这一步：让一个真实句子从 `input_ids` 出发，
沿真实 forward 路径走完 28 个 Decoder Layer，直到 logits 和 Top-K 预测。

本实验采用**两遍结构**：

- **第一遍：观察。** 运行官方 `Qwen3ForCausalLM`，用 hooks 记录每个关键节点的中间状态。
- **第二遍：复现。** 用 PyTorch 基础算子重写每一个模块，参数全部取自真实模型，**不重新初始化**，
  然后与第一遍的记录逐节点对齐。

### 本实验不做的事

- 不研究 tokenizer 如何切词。它只作为入口工具：`自然语言 → tokenizer → input_ids`，从 `input_ids` 开始解剖。
- 不讨论"Qwen3 为什么这样设计"，只弄清"它究竟怎么算"。
- 不使用 KV Cache，只研究完整序列的一次 forward（prefill）。
- 不涉及训练、梯度、采样随机性。

### 真实性原则

不凭记忆猜测计算过程。事实来源按优先级：**本地模型配置 → 本地模型权重 → 实际 Transformers 源码 →
官方模型实际运行结果**。每个重要模块都会记录类名、函数名和源码行号，形成证据链。

## 0. 环境与实验对象核对

正式开始前先确认：我们解剖的是哪一个模型文件，用的是哪一份源码。
后续所有"源码位置"都指向下面这个 `modeling_qwen3.py` 的实际路径。

In [1]:
import inspect
import json
from pathlib import Path

import torch
import transformers
from transformers import AutoModelForCausalLM, AutoTokenizer
from transformers.models.qwen3 import modeling_qwen3 as Q3


def find_project_root() -> Path:
    for directory in (Path.cwd(), *Path.cwd().parents):
        if (directory / 'models' / 'Qwen3-0.6B-Base').is_dir():
            return directory
    raise FileNotFoundError('找不到 models/Qwen3-0.6B-Base')


PROJECT_ROOT = find_project_root()
MODEL_PATH = PROJECT_ROOT / 'models' / 'Qwen3-0.6B-Base'
SOURCE_PATH = Path(Q3.__file__)

print(f'torch:        {torch.__version__}')
print(f'transformers: {transformers.__version__}')
print(f'模型路径:      {MODEL_PATH}')
print(f'源码路径:      {SOURCE_PATH}')
print(f'源码相对位置:  .../{SOURCE_PATH.relative_to(SOURCE_PATH.parents[4])}')

torch:        2.13.0+cu130
transformers: 5.15.1
模型路径:      /home/linjunjie/Workspace/xxdw1/models/Qwen3-0.6B-Base
源码路径:      /home/linjunjie/Workspace/xxdw1/.venv/lib/python3.14/site-packages/transformers/models/qwen3/modeling_qwen3.py
源码相对位置:  .../site-packages/transformers/models/qwen3/modeling_qwen3.py


In [2]:
# 先读磁盘上的 config.json 原文，再读 transformers 解析后的 config 对象。
# 两者不完全一致，这个差异本身就是"以实际运行源码为准"的一个实例。
raw_config = json.loads((MODEL_PATH / 'config.json').read_text())

for key in ['model_type', 'hidden_size', 'num_hidden_layers', 'num_attention_heads',
            'num_key_value_heads', 'head_dim', 'intermediate_size', 'vocab_size',
            'rms_norm_eps', 'rope_theta', 'tie_word_embeddings', 'torch_dtype',
            'attention_bias', 'hidden_act', 'sliding_window']:
    print(f'{key:22s} {raw_config.get(key)}')

model_type             qwen3
hidden_size            1024
num_hidden_layers      28
num_attention_heads    16
num_key_value_heads    8
head_dim               128
intermediate_size      3072
vocab_size             151936
rms_norm_eps           1e-06
rope_theta             1000000
tie_word_embeddings    True
torch_dtype            bfloat16
attention_bias         False
hidden_act             silu
sliding_window         None


### 0.1 关于 dtype 的决定

权重在磁盘上是 `bfloat16`。但本实验**统一加载为 `float32`**，理由来自源码本身：

- `Qwen3RMSNorm.forward` 内部强制 `.to(torch.float32)` 再算方差；
- `eager_attention_forward` 里 softmax 指定 `dtype=torch.float32`；
- `Qwen3RotaryEmbedding.forward` 用 `maybe_autocast(enabled=False)` 强制 float32 算 cos/sin。

也就是说，关键数值路径本来就在 float32 上。统一到 float32 后，官方与我们的复现可以用
`atol=1e-6` 这样的紧公差对齐；若留在 bf16，公差要放宽到 1e-2 量级，验证就失去意义了。

同时指定 `attn_implementation='eager'`，原因是默认的 sdpa 路径**不返回 attention weights**，
且 causal mask 会是 `None`（靠 `is_causal` 标志隐式处理），我们就没有 mask 张量可观察。

In [3]:
DTYPE = torch.float32
DEVICE = 'cpu'

# 全程关闭梯度。本实验只研究推理，不涉及训练。
# 注意这里用 set_grad_enabled 而不是 torch.inference_mode()：
# inference_mode 产生的张量不能再参与后续（被 autograd 追踪的）计算，
# 而我们要用第一遍抓到的张量喂给第二遍的复现代码。
torch.set_grad_enabled(False)

tokenizer = AutoTokenizer.from_pretrained(MODEL_PATH)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_PATH,
    dtype=DTYPE,
    attn_implementation='eager',
).to(DEVICE).eval()

config = model.config
print(f'模型类:        {type(model).__name__}')
print(f'device:       {next(model.parameters()).device}')
print(f'dtype:        {next(model.parameters()).dtype}')
print(f'attn 实现:     {config._attn_implementation}')
print(f'training 模式: {model.training}   (应为 False)')
print(f'梯度开启:      {torch.is_grad_enabled()}   (应为 False)')
print(f'参数量:        {sum(p.numel() for p in model.parameters()):,}')

Loading weights:   0%|          | 0/310 [00:00<?, ?it/s]

模型类:        Qwen3ForCausalLM
device:       cpu
dtype:        torch.float32
attn 实现:     eager
training 模式: False   (应为 False)
梯度开启:      False   (应为 False)
参数量:        596,049,920


### 0.2 config.json 与 config 对象的一处差异

`config.json` 里 `rope_theta` 是顶层字段，但当前版本的 transformers 把它收进了
`config.rope_parameters` 字典。直接访问 `config.rope_theta` 取不到值。

这类差异正是为什么我们不能凭记忆写代码：下面 RoPE 的实现必须从 `config.rope_parameters` 取
`rope_theta`，才和真实运行的源码一致。

In [4]:
print(f'config.rope_parameters       = {config.rope_parameters}')
print(f"hasattr(config, 'rope_theta') = {hasattr(config, 'rope_theta')}")
print(f'config.layer_types 唯一值     = {set(config.layer_types)}')
print(f'model.model.has_sliding_layers = {model.model.has_sliding_layers}')

config.rope_parameters       = {'rope_theta': 1000000, 'rope_type': 'default'}
hasattr(config, 'rope_theta') = False
config.layer_types 唯一值     = {'full_attention'}
model.model.has_sliding_layers = False


In [5]:
# 后续反复使用的维度常量，全部从 config 读取，不写死数字。
N_LAYERS = config.num_hidden_layers          # 28
HIDDEN = config.hidden_size                  # 1024
N_HEADS = config.num_attention_heads         # 16   query 头数
N_KV_HEADS = config.num_key_value_heads      # 8    key/value 头数
HEAD_DIM = config.head_dim                   # 128
N_REP = N_HEADS // N_KV_HEADS                # 2    每个 kv 头被几个 q 头共享
INTERMEDIATE = config.intermediate_size      # 3072
VOCAB = config.vocab_size                    # 151936
RMS_EPS = config.rms_norm_eps                # 1e-06
ROPE_THETA = config.rope_parameters['rope_theta']
SCALING = HEAD_DIM ** -0.5                   # attention 缩放系数

print(f'{N_LAYERS = }, {HIDDEN = }, {INTERMEDIATE = }, {VOCAB = }')
print(f'{N_HEADS = }, {N_KV_HEADS = }, {HEAD_DIM = }, {N_REP = }')
print(f'{RMS_EPS = }, {ROPE_THETA = }, SCALING = {SCALING:.17f}')
print()
print(f'注意 N_HEADS * HEAD_DIM = {N_HEADS * HEAD_DIM} != HIDDEN = {HIDDEN}')
print('    q_proj 会把 1024 维升到 2048 维，o_proj 再降回 1024。')
print(f'    而 N_KV_HEADS * HEAD_DIM = {N_KV_HEADS * HEAD_DIM}，k/v 恰好保持 1024。')

N_LAYERS = 28, HIDDEN = 1024, INTERMEDIATE = 3072, VOCAB = 151936
N_HEADS = 16, N_KV_HEADS = 8, HEAD_DIM = 128, N_REP = 2
RMS_EPS = 1e-06, ROPE_THETA = 1000000, SCALING = 0.08838834764831845

注意 N_HEADS * HEAD_DIM = 2048 != HIDDEN = 1024
    q_proj 会把 1024 维升到 2048 维，o_proj 再降回 1024。
    而 N_KV_HEADS * HEAD_DIM = 1024，k/v 恰好保持 1024。


### 0.3 证据链：本实验涉及的源码位置

下面的行号从实际加载的模块动态读取，不是手抄的。后续每个模块解剖时会再次引用。

In [6]:
def source_location(obj) -> str:
    """返回对象在源码中的文件名与起止行号。装饰器包装过的对象会回退到 __wrapped__。"""
    target = inspect.unwrap(obj)
    try:
        lines, start = inspect.getsourcelines(target)
    except (OSError, TypeError) as exc:
        return f'<无法定位: {exc}>'
    return f'{Path(inspect.getfile(target)).name}:{start}-{start + len(lines) - 1}'


EVIDENCE = [
    ('RMSNorm',          Q3.Qwen3RMSNorm),
    ('  └ forward',      Q3.Qwen3RMSNorm.forward),
    ('MLP',              Q3.Qwen3MLP),
    ('  └ forward',      Q3.Qwen3MLP.forward),
    ('RotaryEmbedding',  Q3.Qwen3RotaryEmbedding),
    ('  └ forward',      Q3.Qwen3RotaryEmbedding.forward),
    ('rotate_half',      Q3.rotate_half),
    ('apply_rotary_pos_emb', Q3.apply_rotary_pos_emb),
    ('repeat_kv',        Q3.repeat_kv),
    ('eager_attention_forward', Q3.eager_attention_forward),
    ('Attention',        Q3.Qwen3Attention),
    ('  └ forward',      Q3.Qwen3Attention.forward),
    ('DecoderLayer',     Q3.Qwen3DecoderLayer),
    ('  └ forward',      Q3.Qwen3DecoderLayer.forward),
    ('Qwen3Model',       Q3.Qwen3Model),
    ('  └ forward',      Q3.Qwen3Model.forward),
    ('Qwen3ForCausalLM', Q3.Qwen3ForCausalLM),
    ('  └ forward',      Q3.Qwen3ForCausalLM.forward),
]

for name, obj in EVIDENCE:
    print(f'{name:24s} {source_location(obj)}')

RMSNorm                  modeling_qwen3.py:49-67
  └ forward              modeling_qwen3.py:59-64
MLP                      modeling_qwen3.py:70-83
  └ forward              modeling_qwen3.py:81-83
RotaryEmbedding          modeling_qwen3.py:86-137
  └ forward              modeling_qwen3.py:124-137
rotate_half              modeling_qwen3.py:140-144
apply_rotary_pos_emb     modeling_qwen3.py:147-170
repeat_kv                modeling_qwen3.py:173-182
eager_attention_forward  modeling_qwen3.py:185-207
Attention                modeling_qwen3.py:210-280
  └ forward              modeling_qwen3.py:241-280
DecoderLayer             modeling_qwen3.py:283-323
  └ forward              modeling_qwen3.py:294-323
Qwen3Model               modeling_qwen3.py:345-427
  └ forward              modeling_qwen3.py:364-427
Qwen3ForCausalLM         modeling_qwen3.py:430-507
  └ forward              modeling_qwen3.py:446-507


## 1. 入口：tokenizer 只负责把文字变成 input_ids

```text
自然语言
   ↓
tokenizer          ← 本实验不解剖它的内部
   ↓
input_ids          ← 解剖从这里开始
   ↓
Qwen3 前向传播
```

选这句话作为实验输入，是因为它在上下文里给出了 `X 是 Y 的首都` 的模式，
模型只要沿模式补全就应该预测出 `首都`。序列短（9 个 token），
后面 9×9 的 attention 矩阵可以整个打印出来看。

In [7]:
PROMPT = '北京是中国的首都，巴黎是法国的'

encoded = tokenizer(PROMPT, return_tensors='pt')
input_ids = encoded.input_ids.to(DEVICE)
BATCH, SEQ = input_ids.shape

print(f'原文: {PROMPT}')
print(f'input_ids shape: {tuple(input_ids.shape)}   (B={BATCH}, S={SEQ})')
print(f'input_ids: {input_ids[0].tolist()}')
print()
print(f'{"pos":>3s}  {"id":>7s}  {"token":<12s}  decode')
for position, token_id in enumerate(input_ids[0].tolist()):
    token = tokenizer.convert_ids_to_tokens([token_id])[0]
    print(f'{position:>3d}  {token_id:>7d}  {token:<12s}  {tokenizer.decode([token_id])!r}')

原文: 北京是中国的首都，巴黎是法国的
input_ids shape: (1, 9)   (B=1, S=9)
input_ids: [68990, 105196, 9370, 106114, 3837, 106004, 20412, 104328, 9370]

pos       id  token         decode
  0    68990  åĮĹäº¬        '北京'
  1   105196  æĺ¯ä¸ŃåĽ½     '是中国'
  2     9370  çļĦ           '的'
  3   106114  é¦ĸéĥ½        '首都'
  4     3837  ï¼Į           '，'
  5   106004  å·´é»İ        '巴黎'
  6    20412  æĺ¯           '是'
  7   104328  æ³ķåĽ½        '法国'
  8     9370  çļĦ           '的'


上面 `token` 列出现的 `åĮĹäº¬` 这类乱码是 byte-level BPE 的字节表示形式，`decode` 列才是可读文本。
这属于 tokenizer 内部机制，本实验不展开。我们只需要 `input_ids` 这 9 个整数。

# 第一遍：观察真实模型

这一遍完全不写自己的计算。目标是运行官方 forward，并把每个关键节点的中间状态**原样记录下来**，
作为第二遍复现时的比对基准。

记录手段有两类：

1. **官方接口**：`output_hidden_states=True` 给出每层输出，`output_attentions=True` 给出 attention 权重；
2. **forward hooks**：官方接口不暴露的更细的节点（q_proj 输出、q_norm 输出、MLP 中间态、causal mask、
   RoPE 的 cos/sin 等），用 hook 抓。

## 2. 挂 hooks

`register_forward_hook(..., with_kwargs=True)` 能同时拿到位置参数、关键字参数和返回值。
Decoder Layer 的 `attention_mask` 和 `position_embeddings` 是以关键字传入的，
所以必须用 `with_kwargs=True` 才抓得到。

In [8]:
from collections import OrderedDict

# 需要抓输出的子模块后缀。前缀 layers.N. 会自动展开到 28 层。
LEAF_SUFFIXES = [
    'input_layernorm',
    'self_attn.q_proj', 'self_attn.k_proj', 'self_attn.v_proj',
    'self_attn.q_norm', 'self_attn.k_norm', 'self_attn.o_proj',
    'post_attention_layernorm',
    'mlp.gate_proj', 'mlp.up_proj', 'mlp.down_proj',
]


class Recorder:
    """记录指定模块的输入/输出张量。"""

    def __init__(self):
        self.out = OrderedDict()    # name -> 输出张量
        self.kw = OrderedDict()     # name -> 关键字参数
        self._handles = []

    def _make(self, name, keep_kwargs):
        def hook(module, args, kwargs, output):
            self.out[name] = output
            if keep_kwargs:
                self.kw[name] = kwargs
        return hook

    def watch(self, module, name, keep_kwargs=False):
        handle = module.register_forward_hook(
            self._make(name, keep_kwargs), with_kwargs=True)
        self._handles.append(handle)

    def remove(self):
        for handle in self._handles:
            handle.remove()
        self._handles.clear()

In [9]:
recorder = Recorder()
named = dict(model.named_modules())

recorder.watch(model.model.embed_tokens, 'embed_tokens')
recorder.watch(model.model.rotary_emb, 'rotary_emb')
recorder.watch(model.model.norm, 'final_norm')
recorder.watch(model.lm_head, 'lm_head')

for layer_index in range(N_LAYERS):
    prefix = f'model.layers.{layer_index}'
    short = f'layers.{layer_index}'
    # Decoder Layer 本身要 keep_kwargs，才能拿到 attention_mask 与 position_embeddings
    recorder.watch(named[prefix], short, keep_kwargs=True)
    recorder.watch(named[f'{prefix}.self_attn'], f'{short}.self_attn')
    recorder.watch(named[f'{prefix}.mlp'], f'{short}.mlp')
    for suffix in LEAF_SUFFIXES:
        recorder.watch(named[f'{prefix}.{suffix}'], f'{short}.{suffix}')

print(f'共挂载 {len(recorder._handles)} 个 hook')

共挂载 396 个 hook


In [10]:
with torch.no_grad():
    official = model(
        input_ids,
        output_hidden_states=True,
        output_attentions=True,
        use_cache=False,
    )

recorder.remove()
print('forward 完成，hooks 已摘除')
print()
print(f'logits          {tuple(official.logits.shape)}')
print(f'hidden_states   {len(official.hidden_states)} 个 × {tuple(official.hidden_states[0].shape)}')
print(f'attentions      {len(official.attentions)} 个 × {tuple(official.attentions[0].shape)}')
print(f'past_key_values {official.past_key_values}   (未用 KV Cache)')

forward 完成，hooks 已摘除

logits          (1, 9, 151936)
hidden_states   29 个 × (1, 9, 1024)
attentions      28 个 × (1, 16, 9, 9)
past_key_values None   (未用 KV Cache)


`hidden_states` 有 **29** 个而不是 28 个：第 0 个是 embedding 的输出（还没进任何 Layer），
之后每个 Decoder Layer 贡献一个。所以 `hidden_states[i]` 是"第 i 层的输入"，
`hidden_states[i+1]` 是"第 i 层的输出"。

注意它们的形状**完全一样**，都是 `[1, 9, 1024]`。这是残差结构的直接后果：
每个 Layer 都是一个保形函数，不管内部把维度升到 2048 还是 3072，出口必须回到 1024。

## 3. 两个观察工具

**Shape Trace**：每个计算节点记录 `输入 shape → 操作 → 输出 shape`。这是本实验的核心记录之一。

**观察窗口**：完整张量全部保存，但打印时只看局部切片，避免 notebook 变成数字的海洋。
不同计算用不同的窗口，例如 hidden state 看 `[0, position, :8]`，attention 权重看整个 `[0, head]` 矩阵。

In [11]:
class ShapeTrace:
    """按计算顺序累积 shape 变化记录。"""

    def __init__(self, title):
        self.title = title
        self.rows = []

    def add(self, operation, out_tensor, in_shape=None, note=''):
        shape = tuple(out_tensor.shape) if torch.is_tensor(out_tensor) else out_tensor
        self.rows.append((operation, in_shape, shape, note))
        return out_tensor

    def show(self):
        width = max(len(r[0]) for r in self.rows)
        print(f'{self.title}')
        print('─' * (width + 46))
        for operation, in_shape, out_shape, note in self.rows:
            arrow = f'{tuple(in_shape)} → ' if in_shape else ''
            line = f'{operation:<{width}s}  {arrow}{out_shape}'
            print(f'{line}{"   # " + note if note else ""}')


def peek(tensor, label, position=-1, width=6, dim_note=''):
    """打印张量的形状与一个小窗口的实际数值。"""
    flat = tensor if tensor.dim() <= 2 else tensor[0]
    if tensor.dim() == 3:      # [B, S, H]
        window = tensor[0, position, :width]
        where = f'[0, {position}, :{width}]'
    elif tensor.dim() == 4:    # [B, heads, S, D]
        window = tensor[0, 0, position, :width]
        where = f'[0, 0, {position}, :{width}]'
    else:
        window = flat.flatten()[:width]
        where = f'flat[:{width}]'
    values = ', '.join(f'{v:+.4f}' for v in window.tolist())
    note = f'  {dim_note}' if dim_note else ''
    print(f'{label:<28s} {str(tuple(tensor.shape)):<22s}{note}')
    print(f'{"":<28s} {where} = [{values}]')

In [12]:
def compare(mine, official_tensor, label, atol=1e-6, rtol=1e-5, quiet=False):
    """对齐检查：allclose + 最大/平均绝对误差 + 相对误差。返回 (通过, 最大误差)。"""
    mine = mine.float()
    ref = official_tensor.float()
    assert mine.shape == ref.shape, f'{label}: shape 不一致 {mine.shape} vs {ref.shape}'
    diff = (mine - ref).abs()
    max_abs = diff.max().item()
    mean_abs = diff.mean().item()
    scale = ref.abs().max().item()
    rel = max_abs / scale if scale > 0 else 0.0
    ok = torch.allclose(mine, ref, atol=atol, rtol=rtol)
    if not quiet:
        flag = '✓' if ok else '✗'
        print(f'{flag} {label:<34s} max_abs={max_abs:.3e}  mean_abs={mean_abs:.3e}  '
              f'max_rel={rel:.3e}')
    return ok, max_abs


VERIFICATIONS = []      # (名称, 是否通过, 最大绝对误差)


def verify(mine, official_tensor, label, **kwargs):
    ok, max_abs = compare(mine, official_tensor, label, **kwargs)
    VERIFICATIONS.append((label, ok, max_abs))
    return ok

## 4. 顶层 Shape Trace：一次 forward 的骨架

先不打开任何模块，只看数据在五个顶层阶段之间的形状变化。这是实验 1 那张图的数值版本。

In [13]:
trace = ShapeTrace('顶层 forward shape trace')
trace.add('input_ids', input_ids, note='B=1, S=9 的整数张量')
trace.add('embed_tokens', recorder.out['embed_tokens'],
          in_shape=input_ids.shape, note='查表：行号 → 1024 维向量')
for layer_index in [0, 1, 27]:
    tensor = recorder.out[f'layers.{layer_index}']
    label = f'  layers.{layer_index}' + ('' if layer_index != 1 else '  ⋯ 中间 26 层省略')
    trace.add(label, tensor, in_shape=(BATCH, SEQ, HIDDEN), note='保形')
trace.add('final_norm', recorder.out['final_norm'],
          in_shape=(BATCH, SEQ, HIDDEN), note='RMSNorm，不改形状')
trace.add('lm_head', official.logits,
          in_shape=(BATCH, SEQ, HIDDEN), note='1024 → 151936 词表打分')
trace.show()

顶层 forward shape trace
─────────────────────────────────────────────────────────────────────
input_ids                (1, 9)   # B=1, S=9 的整数张量
embed_tokens             (1, 9) → (1, 9, 1024)   # 查表：行号 → 1024 维向量
  layers.0               (1, 9, 1024) → (1, 9, 1024)   # 保形
  layers.1  ⋯ 中间 26 层省略  (1, 9, 1024) → (1, 9, 1024)   # 保形
  layers.27              (1, 9, 1024) → (1, 9, 1024)   # 保形
final_norm               (1, 9, 1024) → (1, 9, 1024)   # RMSNorm，不改形状
lm_head                  (1, 9, 1024) → (1, 9, 151936)   # 1024 → 151936 词表打分


# 第二遍：自己复现

原则是**能自己实现就自己实现**，但不重复制造 PyTorch。

**自己实现**：Embedding、Linear、RMSNorm、RoPE、GQA、Attention、causal mask、MLP、Residual、Decoder Layer。

**直接用 PyTorch 基础算子**：`matmul`、`reshape`、`view`、`transpose`、`softmax`、`exp`、`rsqrt`、`sigmoid` 等。

Linear 和 RMSNorm 是第一次完整实现，之后所有地方复用自己的这一份实现。

## 5. 最基础的两块积木

### 5.1 Linear（无 bias）

`config.attention_bias = False`，且 MLP 的三个投影在源码里都是 `bias=False`，
所以本模型**所有** Linear 都没有 bias。计算就是一次矩阵乘：

$$\mathrm{Linear}(x) = x W^\top$$

`nn.Linear` 的权重形状是 `[out_features, in_features]`，所以要转置后右乘。

In [14]:
def my_linear(x, weight):
    """无 bias 的线性层。weight: [out_features, in_features]"""
    return x @ weight.T


# 用真实权重验证：拿 layer 0 的 q_proj 对比官方输出
_probe_in = recorder.out['layers.0.input_layernorm']
_probe_w = model.model.layers[0].self_attn.q_proj.weight
print(f'q_proj.weight shape: {tuple(_probe_w.shape)}  (out=2048, in=1024)')
verify(my_linear(_probe_in, _probe_w), recorder.out['layers.0.self_attn.q_proj'],
       'my_linear vs q_proj')

q_proj.weight shape: (2048, 1024)  (out=2048, in=1024)
✓ my_linear vs q_proj                max_abs=0.000e+00  mean_abs=0.000e+00  max_rel=0.000e+00


True

### 5.2 RMSNorm

源码 `Qwen3RMSNorm.forward`（见 §0.3 证据链）的计算顺序是：

$$\bar{x} = x \cdot \frac{1}{\sqrt{\frac{1}{d}\sum_{i=1}^{d} x_i^2 + \epsilon}}, \qquad
\mathrm{RMSNorm}(x) = w \odot \bar{x}$$

严格按代码顺序，有三个容易写错的细节：

1. **先转 float32 再算**，最后才转回输入 dtype；
2. `weight` 的乘法发生在**转回 dtype 之后**，不是在 float32 里乘；
3. 用 `rsqrt(variance + eps)`，eps 在**根号内部**，不是外部；
4. 这里的 variance 是**平方的均值**，不减均值（这是 RMSNorm 与 LayerNorm 的区别）。

In [15]:
def my_rmsnorm(x, weight, eps=RMS_EPS):
    """严格按 Qwen3RMSNorm.forward 的顺序实现。归一化沿最后一维进行。"""
    input_dtype = x.dtype
    x = x.to(torch.float32)                              # 1. 升到 float32
    variance = x.pow(2).mean(-1, keepdim=True)           # 2. 平方的均值，不减均值
    x = x * torch.rsqrt(variance + eps)                  # 3. eps 在根号内
    return weight * x.to(input_dtype)                    # 4. 先转回 dtype，再乘 weight


_layer0 = model.model.layers[0]
verify(my_rmsnorm(official.hidden_states[0], _layer0.input_layernorm.weight),
       recorder.out['layers.0.input_layernorm'], 'my_rmsnorm vs input_layernorm')
# post_attention_layernorm 的输入是"第一次残差之后"的中间态，
# 它不是任何模块的直接输出，得自己拼：layer 输入 + self_attn 输出。
_after_attn = official.hidden_states[0] + recorder.out['layers.0.self_attn'][0]
verify(my_rmsnorm(_after_attn, _layer0.post_attention_layernorm.weight),
       recorder.out['layers.0.post_attention_layernorm'],
       'my_rmsnorm vs post_attn_norm')

✓ my_rmsnorm vs input_layernorm      max_abs=0.000e+00  mean_abs=0.000e+00  max_rel=0.000e+00
✓ my_rmsnorm vs post_attn_norm       max_abs=0.000e+00  mean_abs=0.000e+00  max_rel=0.000e+00


True

### 5.3 SiLU 激活

`config.hidden_act = 'silu'`，即 $\mathrm{SiLU}(x) = x \cdot \sigma(x) = \dfrac{x}{1 + e^{-x}}$。
我们用 `torch.sigmoid` 这个基础算子拼出来，不调 `nn.functional.silu`。

In [16]:
def my_silu(x):
    return x * torch.sigmoid(x)


_t = torch.randn(4, 8, dtype=DTYPE)
verify(my_silu(_t), torch.nn.functional.silu(_t), 'my_silu vs F.silu')

✓ my_silu vs F.silu                  max_abs=1.192e-07  mean_abs=1.118e-08  max_rel=5.280e-08


True

## 6. Embedding：一次按行查表

```text
              input_ids [B, S]              整数，取值范围 [0, 151936)
                   │
                   │  以 token id 作为行号，从权重矩阵取行
                   ▼
   embed_tokens.weight [151936, 1024]
                   │
                   ▼
        inputs_embeds [B, S, 1024]
```

`nn.Embedding` 的 forward 没有矩阵乘、没有激活，就是一次索引。所以"自己实现"它等价于
`weight[input_ids]`。下面顺带验证一件事：**这张表就是最后 lm_head 用的那张表**。

In [17]:
def my_embedding(ids, weight):
    """按行号取行。等价于 nn.Embedding.forward（无 padding_idx 参与时）。"""
    return weight[ids]


EMBED_WEIGHT = model.model.embed_tokens.weight
my_embeds = my_embedding(input_ids, EMBED_WEIGHT)

verify(my_embeds, recorder.out['embed_tokens'], 'my_embedding vs embed_tokens')
verify(my_embeds, official.hidden_states[0], 'my_embedding vs hidden_states[0]')

print()
print(f'embed_tokens.weight    {tuple(EMBED_WEIGHT.shape)}')
print(f'lm_head.weight         {tuple(model.lm_head.weight.shape)}')
print(f'是同一个张量对象:       {EMBED_WEIGHT is model.lm_head.weight}')
print(f'共享同一块内存:         {EMBED_WEIGHT.data_ptr() == model.lm_head.weight.data_ptr()}')
print(f'_tied_weights_keys:    {type(model)._tied_weights_keys}')
print()
_embed_params = VOCAB * HIDDEN
_total = sum(p.numel() for p in model.parameters())
print(f'这张表的参数量: {_embed_params:,} = 总参数 {_total:,} 的 {_embed_params / _total:.1%}')

✓ my_embedding vs embed_tokens       max_abs=0.000e+00  mean_abs=0.000e+00  max_rel=0.000e+00
✓ my_embedding vs hidden_states[0]   max_abs=0.000e+00  mean_abs=0.000e+00  max_rel=0.000e+00

embed_tokens.weight    (151936, 1024)
lm_head.weight         (151936, 1024)
是同一个张量对象:       True
共享同一块内存:         True
_tied_weights_keys:    {'lm_head.weight': 'model.embed_tokens.weight'}

这张表的参数量: 155,582,464 = 总参数 596,049,920 的 26.1%


`tie_word_embeddings: true` 的含义在这里变得具体：**同一张 `[151936, 1024]` 的矩阵被用了两次**。

- 入口（Embedding）当字典查：行号 → 向量；
- 出口（LM Head）当打分器用：把 hidden state 和 151936 行逐行做内积。

这也是为什么 `model.safetensors` 里存了 310 个张量，却**没有** `lm_head.weight`——它根本不需要存。
这一张表占了模型总参数的四分之一以上。

In [18]:
# 逐个 token 确认"查表"就是字面意义的取行
for position in [0, SEQ - 1]:
    token_id = input_ids[0, position].item()
    row = EMBED_WEIGHT[token_id]
    got = my_embeds[0, position]
    same = torch.equal(row, got)
    text = tokenizer.decode([token_id])
    print(f'pos {position}  id={token_id:<7d} {text!r:<6s} '
          f'embeds[0,{position}] 是否等于 weight[{token_id}]: {same}')
    print(f'         前 6 维 = [{", ".join(f"{v:+.5f}" for v in row[:6].tolist())}]')

pos 0  id=68990   '北京'   embeds[0,0] 是否等于 weight[68990]: True
         前 6 维 = [-0.03198, -0.04858, +0.03345, +0.04492, -0.03296, -0.04077]
pos 8  id=9370    '的'    embeds[0,8] 是否等于 weight[9370]: True
         前 6 维 = [-0.01941, -0.00482, -0.04858, -0.01636, -0.00824, +0.02405]


## 7. 进入 Layer 之前：两样全局预备件

看源码 `Qwen3Model.forward` 会发现，进入 28 层循环之前先算好了两样东西，
然后**原样传给每一层**，28 层共用，不重复计算：

```text
inputs_embeds
     │
     ├──→ position_ids ──→ rotary_emb ──→ (cos, sin)   ─┐
     │                                                  ├─→ 传给全部 28 层
     └──→ create_causal_mask ──→ attention_mask        ─┘
```

这一点很容易被忽略：RoPE 的 cos/sin **不在 Attention 内部计算**，而是模型级别算一次。

In [19]:
# Qwen3Model.forward 里的 position_ids：没传就是 0..S-1
my_position_ids = torch.arange(SEQ, device=DEVICE).unsqueeze(0)
print(f'position_ids {tuple(my_position_ids.shape)} = {my_position_ids[0].tolist()}')

# 官方实际用的 position_ids（从 layer 0 的 kwargs 抓到）
official_position_ids = recorder.kw['layers.0']['position_ids']
verify(my_position_ids.float(), official_position_ids.float(), 'my_position_ids')

position_ids (1, 9) = [0, 1, 2, 3, 4, 5, 6, 7, 8]
✓ my_position_ids                    max_abs=0.000e+00  mean_abs=0.000e+00  max_rel=0.000e+00


True

### 7.1 RoPE：cos / sin 表

RoPE 的目标是把"位置"编码成一个旋转。频率向量（`inv_freq`）只依赖 head_dim 和 theta：

$$\theta_j = \frac{1}{\text{base}^{\,2j/d}}, \qquad j = 0, 1, \dots, \frac{d}{2}-1$$

其中 $d = 128$（head_dim），base = `rope_theta` = 1000000。注意源码里
`torch.arange(0, dim, 2) / dim` 得到的是 $2j/d$，所以 `inv_freq` 长度是 64。

然后与位置做外积，再把结果**复制一份拼接**成 128 维：

$$\text{freqs}[p, j] = p \cdot \theta_j \quad (\text{形状 } S \times 64), \qquad
\text{emb} = [\text{freqs}, \text{freqs}] \quad (S \times 128)$$

$$\cos = \cos(\text{emb}), \qquad \sin = \sin(\text{emb})$$

拼接这一步是为了配合后面 `rotate_half` 的实现方式。

In [20]:
def my_rope_tables(position_ids, head_dim=HEAD_DIM, base=ROPE_THETA, dtype=DTYPE):
    """复现 Qwen3RotaryEmbedding：返回 (cos, sin)，形状 [B, S, head_dim]。"""
    # inv_freq: [head_dim/2]，全程 float32
    exponent = torch.arange(0, head_dim, 2, dtype=torch.float32) / head_dim
    inv_freq = 1.0 / (base ** exponent)

    # 外积：[B, head_dim/2, 1] @ [B, 1, S] -> [B, head_dim/2, S] -> transpose -> [B, S, head_dim/2]
    inv_freq_expanded = inv_freq[None, :, None].expand(position_ids.shape[0], -1, 1)
    positions_expanded = position_ids[:, None, :].float()
    freqs = (inv_freq_expanded @ positions_expanded).transpose(1, 2)

    emb = torch.cat((freqs, freqs), dim=-1)      # [B, S, head_dim]
    return emb.cos().to(dtype), emb.sin().to(dtype), inv_freq


my_cos, my_sin, my_inv_freq = my_rope_tables(my_position_ids)
official_cos, official_sin = recorder.kw['layers.0']['position_embeddings']

print(f'inv_freq {tuple(my_inv_freq.shape)}  前4 = '
      f'[{", ".join(f"{v:.3e}" for v in my_inv_freq[:4].tolist())}]')
print(f'         后4 = [{", ".join(f"{v:.3e}" for v in my_inv_freq[-4:].tolist())}]')
print(f'cos/sin  {tuple(my_cos.shape)}')
print()
verify(my_cos, official_cos, 'my_rope cos')
verify(my_sin, official_sin, 'my_rope sin')
verify(my_inv_freq, model.model.rotary_emb.inv_freq, 'my_inv_freq')

inv_freq (64,)  前4 = [1.000e+00, 8.058e-01, 6.494e-01, 5.233e-01]
         后4 = [2.371e-06, 1.911e-06, 1.540e-06, 1.241e-06]
cos/sin  (1, 9, 128)

✓ my_rope cos                        max_abs=0.000e+00  mean_abs=0.000e+00  max_rel=0.000e+00
✓ my_rope sin                        max_abs=0.000e+00  mean_abs=0.000e+00  max_rel=0.000e+00
✓ my_inv_freq                        max_abs=0.000e+00  mean_abs=0.000e+00  max_rel=0.000e+00


True

In [21]:
# 观察窗口：cos 表的前 4 个位置 × 前 3 个频率通道
print('cos[0, position, channel] 的一角：')
print(f'{"":>8s}' + ''.join(f'ch{c:<10d}' for c in range(3)))
for position in range(4):
    row = ''.join(f'{my_cos[0, position, c].item():+11.6f}' for c in range(3))
    print(f'pos {position:<3d} {row}')
print()
print('ch0 频率最高（相邻位置差异大），高编号通道频率极低（长距离才有区分度）：')
for channel in [0, 32, 63]:
    values = ', '.join(f'{my_cos[0, p, channel].item():+.6f}' for p in range(SEQ))
    print(f'  ch{channel:<3d} 沿位置变化 = [{values}]')

cos[0, position, channel] 的一角：
        ch0         ch1         ch2         
pos 0     +1.000000  +1.000000  +1.000000
pos 1     +0.540302  +0.692504  +0.796458
pos 2     -0.416147  -0.040877  +0.268690
pos 3     -0.989992  -0.749119  -0.368457

ch0 频率最高（相邻位置差异大），高编号通道频率极低（长距离才有区分度）：
  ch0   沿位置变化 = [+1.000000, +0.540302, -0.416147, -0.989992, -0.653644, +0.283662, +0.960170, +0.753902, -0.145500]
  ch32  沿位置变化 = [+1.000000, +1.000000, +0.999998, +0.999996, +0.999992, +0.999987, +0.999982, +0.999976, +0.999968]
  ch63  沿位置变化 = [+1.000000, +1.000000, +1.000000, +1.000000, +1.000000, +1.000000, +1.000000, +1.000000, +1.000000]


### 7.2 Causal Mask

因果掩码保证位置 $i$ 只能看到 $j \le i$ 的位置。eager 路径下它是一个**加性** mask：
允许的位置填 0，禁止的位置填一个极小的数（`torch.finfo(float32).min`），
加到 attention 分数上之后，softmax 会把这些位置压到 0。

```text
        j=0   1    2   ...          ← 被看的位置（key）
 i=0  [  0  -inf -inf ...  ]
 i=1  [  0    0  -inf ...  ]        ← 看的位置（query）
 i=2  [  0    0    0  ...  ]
```

In [22]:
def my_causal_mask(seq_len, dtype=DTYPE, device=DEVICE):
    """加性因果掩码，形状 [1, 1, S, S]，可广播到 [B, heads, S, S]。"""
    blocked = torch.finfo(dtype).min
    positions = torch.arange(seq_len, device=device)
    # query 位置 i 只能看 key 位置 j <= i
    allowed = positions[:, None] >= positions[None, :]
    mask = torch.where(allowed, torch.zeros((), dtype=dtype, device=device),
                       torch.full((), blocked, dtype=dtype, device=device))
    return mask[None, None, :, :]


my_mask = my_causal_mask(SEQ)
official_mask = recorder.kw['layers.0']['attention_mask']

print(f'官方 mask: {tuple(official_mask.shape)}  dtype={official_mask.dtype}')
print(f'屏蔽值:    {official_mask.min().item():.6e}')
print(f'等于 torch.finfo(float32).min: '
      f'{official_mask.min().item() == torch.finfo(torch.float32).min}')
print()
verify(my_mask, official_mask, 'my_causal_mask')
print()
print('mask[0,0] 的 0/-inf 结构（0 表示可见，· 表示屏蔽）：')
for i in range(SEQ):
    row = ' '.join('0' if official_mask[0, 0, i, j] == 0 else '·' for j in range(SEQ))
    print(f'  i={i}  {row}')

官方 mask: (1, 1, 9, 9)  dtype=torch.float32
屏蔽值:    -3.402823e+38
等于 torch.finfo(float32).min: True

✓ my_causal_mask                     max_abs=0.000e+00  mean_abs=0.000e+00  max_rel=0.000e+00

mask[0,0] 的 0/-inf 结构（0 表示可见，· 表示屏蔽）：
  i=0  0 · · · · · · · ·
  i=1  0 0 · · · · · · ·
  i=2  0 0 0 · · · · · ·
  i=3  0 0 0 0 · · · · ·
  i=4  0 0 0 0 0 · · · ·
  i=5  0 0 0 0 0 0 · · ·
  i=6  0 0 0 0 0 0 0 · ·
  i=7  0 0 0 0 0 0 0 0 ·
  i=8  0 0 0 0 0 0 0 0 0


所有 28 层共用同一个 mask 对象和同一组 cos/sin，下面确认这一点。

In [23]:
mask_shared = all(recorder.kw[f'layers.{i}']['attention_mask'] is official_mask
                  for i in range(N_LAYERS))
rope_shared = all(recorder.kw[f'layers.{i}']['position_embeddings'][0] is official_cos
                  for i in range(N_LAYERS))
print(f'28 层共用同一个 attention_mask 对象: {mask_shared}')
print(f'28 层共用同一组 (cos, sin) 对象:      {rope_shared}')

28 层共用同一个 attention_mask 对象: True
28 层共用同一组 (cos, sin) 对象:      True


## 8. 完整解剖 Layer 0

28 层结构完全相同，所以只对第 0 层做逐步解剖。后续 27 层用同样的逻辑批量计算并全部验证，
但不重复写 27 遍说明。

```text
                    Layer Input  [B, S, 1024]
                          │
             ┌────────────┤ residual
             │            ▼
             │      input_layernorm (RMSNorm)      [B, S, 1024]
             │            ▼
             │      ┌─────────────────────────┐
             │      │      Attention          │
             │      │  q/k/v_proj → q/k_norm  │
             │      │  → RoPE → GQA → QKᵀ     │
             │      │  → mask → softmax → ×V  │
             │      │  → o_proj               │
             │      └─────────────────────────┘    [B, S, 1024]
             │            ▼
             └──────────► ⊕  residual add          [B, S, 1024]
                          │
             ┌────────────┤ residual
             │            ▼
             │      post_attention_layernorm       [B, S, 1024]
             │            ▼
             │      ┌─────────────────────────┐
             │      │          MLP            │
             │      │  gate_proj  up_proj     │    [B, S, 3072]
             │      │  SiLU(gate) * up        │
             │      │  down_proj              │    [B, S, 1024]
             │      └─────────────────────────┘
             │            ▼
             └──────────► ⊕  residual add
                          │
                    Layer Output [B, S, 1024]
```

源码依据：`Qwen3DecoderLayer.forward`（§0.3 证据链给出行号）。
注意两个 RMSNorm 都在**子模块之前**（pre-norm），残差加的是**未归一化**的输入。

In [24]:
LAYER = 0
layer = model.model.layers[LAYER]
attn = layer.self_attn
mlp = layer.mlp

t0 = ShapeTrace(f'Layer {LAYER} shape trace')

# ── Layer 输入 ──
x_in = official.hidden_states[LAYER]
t0.add('layer input', x_in, note='= hidden_states[0]，即 embedding 输出')
residual_1 = x_in                      # 残差记住的是未归一化的输入

# ── 8.1 input_layernorm ──
h = my_rmsnorm(x_in, layer.input_layernorm.weight)
t0.add('input_layernorm', h, in_shape=x_in.shape, note='RMSNorm，形状不变')
verify(h, recorder.out[f'layers.{LAYER}.input_layernorm'],
       f'L{LAYER} input_layernorm')
peek(x_in, 'layer input', position=-1)
peek(h, 'after input_layernorm', position=-1)

✓ L0 input_layernorm                 max_abs=0.000e+00  mean_abs=0.000e+00  max_rel=0.000e+00
layer input                  (1, 9, 1024)          
                             [0, -1, :6] = [-0.0194, -0.0048, -0.0486, -0.0164, -0.0082, +0.0240]
after input_layernorm        (1, 9, 1024)          
                             [0, -1, :6] = [-0.1020, -0.1308, -1.1080, -0.4511, -0.0651, +0.5843]


### 8.2 Q / K / V 投影

三个投影的输出维度不一样，这是 GQA（Grouped Query Attention）的起点：

| 投影 | weight 形状 | 输出 | 含义 |
|---|---|---|---|
| `q_proj` | `[2048, 1024]` | `[B, S, 2048]` | 16 个 query 头 × 128 |
| `k_proj` | `[1024, 1024]` | `[B, S, 1024]` | **8** 个 key 头 × 128 |
| `v_proj` | `[1024, 1024]` | `[B, S, 1024]` | **8** 个 value 头 × 128 |

query 头数是 kv 头数的 2 倍。注意 `q_proj` 把 1024 维**升到了 2048**，
比 hidden_size 还大——这是 Qwen3 的选择，`head_dim=128` 是配置里写死的，不是 `hidden_size / n_heads`。

In [25]:
q_flat = my_linear(h, attn.q_proj.weight)
k_flat = my_linear(h, attn.k_proj.weight)
v_flat = my_linear(h, attn.v_proj.weight)

t0.add('q_proj', q_flat, in_shape=h.shape, note='16 头 × 128')
t0.add('k_proj', k_flat, in_shape=h.shape, note='8 头 × 128')
t0.add('v_proj', v_flat, in_shape=h.shape, note='8 头 × 128')

verify(q_flat, recorder.out[f'layers.{LAYER}.self_attn.q_proj'], f'L{LAYER} q_proj')
verify(k_flat, recorder.out[f'layers.{LAYER}.self_attn.k_proj'], f'L{LAYER} k_proj')
verify(v_flat, recorder.out[f'layers.{LAYER}.self_attn.v_proj'], f'L{LAYER} v_proj')

✓ L0 q_proj                          max_abs=0.000e+00  mean_abs=0.000e+00  max_rel=0.000e+00
✓ L0 k_proj                          max_abs=0.000e+00  mean_abs=0.000e+00  max_rel=0.000e+00
✓ L0 v_proj                          max_abs=0.000e+00  mean_abs=0.000e+00  max_rel=0.000e+00


True

### 8.3 reshape 成多头，然后做 q_norm / k_norm

**这一步是 Qwen3 与 Llama 最关键的结构差异。** 源码里这三行把好几个操作压在了一起：

```python
query_states = self.q_norm(self.q_proj(hidden_states).view(hidden_shape)).transpose(1, 2)
key_states   = self.k_norm(self.k_proj(hidden_states).view(hidden_shape)).transpose(1, 2)
value_states = self.v_proj(hidden_states).view(hidden_shape).transpose(1, 2)
```

拆开看，真实顺序是：

```text
q_proj 输出  [B, S, 2048]
    ↓ view(B, S, -1, 128)
             [B, S, 16, 128]
    ↓ q_norm  ← RMSNorm 作用在最后一维 head_dim=128 上，不是 1024！
             [B, S, 16, 128]
    ↓ transpose(1, 2)
             [B, 16, S, 128]
```

三个要点：

1. norm 在 **reshape 之后**做，所以归一化的单位是**每个头的 128 维向量**，不是整个 2048；
2. `q_norm.weight` 的形状是 `[128]`，**16 个头共享同一组 128 个缩放参数**；
3. **V 没有 norm**，只有 Q 和 K 有。

In [26]:
print(f'q_norm.weight 形状: {tuple(attn.q_norm.weight.shape)}  '
      f'← 长度 {HEAD_DIM}，被 {N_HEADS} 个 q 头共享')
print(f'k_norm.weight 形状: {tuple(attn.k_norm.weight.shape)}  '
      f'← 长度 {HEAD_DIM}，被 {N_KV_HEADS} 个 kv 头共享')
print(f'v 有 norm 吗: {hasattr(attn, "v_norm")}')
print()

# view: 把最后一维拆成 (头数, head_dim)
q_heads = q_flat.view(BATCH, SEQ, N_HEADS, HEAD_DIM)
k_heads = k_flat.view(BATCH, SEQ, N_KV_HEADS, HEAD_DIM)
v_heads = v_flat.view(BATCH, SEQ, N_KV_HEADS, HEAD_DIM)
t0.add('q view', q_heads, in_shape=q_flat.shape, note='2048 拆成 16×128')
t0.add('k view', k_heads, in_shape=k_flat.shape, note='1024 拆成 8×128')

# q_norm / k_norm 在 head_dim 上做 RMSNorm
q_normed = my_rmsnorm(q_heads, attn.q_norm.weight)
k_normed = my_rmsnorm(k_heads, attn.k_norm.weight)
t0.add('q_norm', q_normed, in_shape=q_heads.shape, note='在 head_dim=128 上归一化')
t0.add('k_norm', k_normed, in_shape=k_heads.shape, note='V 不做 norm')

verify(q_normed, recorder.out[f'layers.{LAYER}.self_attn.q_norm'], f'L{LAYER} q_norm')
verify(k_normed, recorder.out[f'layers.{LAYER}.self_attn.k_norm'], f'L{LAYER} k_norm')

q_norm.weight 形状: (128,)  ← 长度 128，被 16 个 q 头共享
k_norm.weight 形状: (128,)  ← 长度 128，被 8 个 kv 头共享
v 有 norm 吗: False

✓ L0 q_norm                          max_abs=0.000e+00  mean_abs=0.000e+00  max_rel=0.000e+00
✓ L0 k_norm                          max_abs=0.000e+00  mean_abs=0.000e+00  max_rel=0.000e+00


True

In [27]:
# transpose 把头维提到前面，之后所有 attention 计算都在 [B, heads, S, head_dim] 上做
q = q_normed.transpose(1, 2)
k = k_normed.transpose(1, 2)
v = v_heads.transpose(1, 2)

t0.add('q transpose(1,2)', q, in_shape=q_normed.shape, note='头维提前')
t0.add('k transpose(1,2)', k, in_shape=k_normed.shape)
t0.add('v transpose(1,2)', v, in_shape=v_heads.shape)

print('norm 前后对比（head 0，最后一个位置，前 6 维）：')
peek(q_heads.transpose(1, 2), 'q 未 norm')
peek(q, 'q 已 norm')
print()
print('验证 q_norm 真的是逐头归一化：head 0 的 128 维向量，其均方根应该接近 1')
_h0 = q_heads[0, -1, 0].float()
print(f'  norm 前 RMS = {_h0.pow(2).mean().sqrt().item():.4f}')
_h0n = (q_normed[0, -1, 0] / attn.q_norm.weight).float()
print(f'  norm 后（除掉 weight）RMS = {_h0n.pow(2).mean().sqrt().item():.6f}')

norm 前后对比（head 0，最后一个位置，前 6 维）：
q 未 norm                     (1, 16, 9, 128)       
                             [0, 0, -1, :6] = [+0.0292, +0.1119, -0.0286, -0.1662, -0.0711, +0.1085]
q 已 norm                     (1, 16, 9, 128)       
                             [0, 0, -1, :6] = [+0.5642, +0.5918, +0.0894, -1.2054, -0.7853, +0.7144]

验证 q_norm 真的是逐头归一化：head 0 的 128 维向量，其均方根应该接近 1
  norm 前 RMS = 0.2349
  norm 后（除掉 weight）RMS = 0.999991


### 8.4 应用 RoPE

$$q' = q \odot \cos + \mathrm{rotate\_half}(q) \odot \sin$$

其中 `rotate_half` 把 128 维**前后对半切开**再交叉取负：

$$\mathrm{rotate\_half}([x_1, x_2]) = [-x_2, x_1], \qquad x_1 = x[:64],\ x_2 = x[64:]$$

这与"把相邻两维配成一对做二维旋转"的经典写法在数学上等价，但**维度配对方式不同**：
这里配对的是 $(i, i+64)$，不是 $(2i, 2i+1)$。这也解释了 §7.1 为什么要把 freqs 复制拼接成 128 维——
`cos` 的第 $i$ 维和第 $i+64$ 维是同一个角度。

cos/sin 形状是 `[B, S, 128]`，要 `unsqueeze(1)` 变成 `[B, 1, S, 128]` 才能广播到 `[B, heads, S, 128]`。
**Q 和 K 都要转，V 不转。**

In [28]:
def my_rotate_half(x):
    """把最后一维对半切开，交叉取负。"""
    half = x.shape[-1] // 2
    x1, x2 = x[..., :half], x[..., half:]
    return torch.cat((-x2, x1), dim=-1)


def my_apply_rope(q, k, cos, sin):
    """cos/sin: [B, S, head_dim] -> unsqueeze 到 [B, 1, S, head_dim] 广播。"""
    cos = cos.unsqueeze(1)
    sin = sin.unsqueeze(1)
    q_out = q * cos + my_rotate_half(q) * sin
    k_out = k * cos + my_rotate_half(k) * sin
    return q_out, k_out


verify(my_rotate_half(q), Q3.rotate_half(q), 'my_rotate_half')

q_rope, k_rope = my_apply_rope(q, k, my_cos, my_sin)
t0.add('RoPE(q)', q_rope, in_shape=q.shape, note='形状不变，只旋转')
t0.add('RoPE(k)', k_rope, in_shape=k.shape, note='V 不参与')

_ref_q, _ref_k = Q3.apply_rotary_pos_emb(q, k, official_cos, official_sin)
verify(q_rope, _ref_q, f'L{LAYER} RoPE(q)')
verify(k_rope, _ref_k, f'L{LAYER} RoPE(k)')

✓ my_rotate_half                     max_abs=0.000e+00  mean_abs=0.000e+00  max_rel=0.000e+00
✓ L0 RoPE(q)                         max_abs=0.000e+00  mean_abs=0.000e+00  max_rel=0.000e+00
✓ L0 RoPE(k)                         max_abs=0.000e+00  mean_abs=0.000e+00  max_rel=0.000e+00


True

In [29]:
# RoPE 保长度：旋转不改变向量的模
_before = q[0, 0, -1].float().norm().item()
_after = q_rope[0, 0, -1].float().norm().item()
print(f'RoPE 前后向量模长（head 0, 最后位置）: {_before:.6f} → {_after:.6f}')
print(f'相对变化: {abs(_after - _before) / _before:.2e}   ← 旋转是保长变换')
print()
print('位置 0 的 cos=1, sin=0，所以位置 0 的 q 不应被改变：')
_p0_diff = (q_rope[0, 0, 0] - q[0, 0, 0]).abs().max().item()
print(f'  q_rope[0,0,0] 与 q[0,0,0] 的最大差异 = {_p0_diff:.3e}')

RoPE 前后向量模长（head 0, 最后位置）: 16.679739 → 16.679739
相对变化: 0.00e+00   ← 旋转是保长变换

位置 0 的 cos=1, sin=0，所以位置 0 的 q 不应被改变：
  q_rope[0,0,0] 与 q[0,0,0] 的最大差异 = 0.000e+00


### 8.5 GQA：repeat_kv

16 个 query 头要和 8 个 kv 头对齐。做法是把每个 kv 头**复制 2 份**：

```text
k: [B, 8, S, 128]
     ↓ 插入一个长度 2 的维度并 expand
   [B, 8, 2, S, 128]
     ↓ reshape 合并前两维
   [B, 16, S, 128]
```

复制方式是 `repeat_interleave` 语义：kv 头 0 服务 q 头 0 和 1，kv 头 1 服务 q 头 2 和 3，以此类推。
`expand` 不复制内存，`reshape` 才实际展开。

这一步发生在 **RoPE 之后**。顺序很重要：如果先 repeat 再转 RoPE，就要多算一倍的旋转。

In [30]:
def my_repeat_kv(hidden_states, n_rep):
    """[B, KV, S, D] -> [B, KV*n_rep, S, D]，等价于 repeat_interleave(dim=1)。"""
    batch, n_kv, seq_len, head_dim = hidden_states.shape
    if n_rep == 1:
        return hidden_states
    expanded = hidden_states[:, :, None, :, :].expand(batch, n_kv, n_rep, seq_len, head_dim)
    return expanded.reshape(batch, n_kv * n_rep, seq_len, head_dim)


k_rep = my_repeat_kv(k_rope, N_REP)
v_rep = my_repeat_kv(v, N_REP)

t0.add('repeat_kv(k)', k_rep, in_shape=k_rope.shape, note=f'8 头 × {N_REP} → 16 头')
t0.add('repeat_kv(v)', v_rep, in_shape=v.shape)

verify(k_rep, Q3.repeat_kv(_ref_k, N_REP), f'L{LAYER} repeat_kv(k)')
verify(v_rep, Q3.repeat_kv(v, N_REP), f'L{LAYER} repeat_kv(v)')

print()
print('确认复制的对应关系（q 头 i ← kv 头 i // 2）：')
for q_head in [0, 1, 2, 3, 14, 15]:
    kv_head = q_head // N_REP
    same = torch.equal(k_rep[0, q_head], k_rope[0, kv_head])
    print(f'  k_rep[head {q_head:2d}] == k_rope[head {kv_head}] : {same}')

✓ L0 repeat_kv(k)                    max_abs=0.000e+00  mean_abs=0.000e+00  max_rel=0.000e+00
✓ L0 repeat_kv(v)                    max_abs=0.000e+00  mean_abs=0.000e+00  max_rel=0.000e+00

确认复制的对应关系（q 头 i ← kv 头 i // 2）：
  k_rep[head  0] == k_rope[head 0] : True
  k_rep[head  1] == k_rope[head 0] : True
  k_rep[head  2] == k_rope[head 1] : True
  k_rep[head  3] == k_rope[head 1] : True
  k_rep[head 14] == k_rope[head 7] : True
  k_rep[head 15] == k_rope[head 7] : True


### 8.6 QKᵀ、mask、softmax

$$\text{scores} = \frac{Q K^\top}{\sqrt{d_k}}, \qquad d_k = 128,\ \frac{1}{\sqrt{128}} \approx 0.088388$$

$$A = \mathrm{softmax}(\text{scores} + \text{mask})$$

严格按源码 `eager_attention_forward`：

1. 缩放在 matmul **之后**乘（`torch.matmul(q, k.T) * scaling`），不是先缩放 q；
2. mask 是**加**上去的，不是乘或填充；
3. softmax 指定 `dtype=torch.float32`，算完再转回 q 的 dtype。

shape 变化是整个 Attention 里最需要盯住的一段：
`[B, 16, S, 128] @ [B, 16, 128, S] → [B, 16, S, S]`。序列维度出现了两次，
第一个 S 是"谁在看"，第二个 S 是"看谁"。

In [31]:
scores = torch.matmul(q_rope, k_rep.transpose(2, 3)) * SCALING
t0.add('QKᵀ * scaling', scores, in_shape=q_rope.shape,
       note='[B,16,S,128] @ [B,16,128,S]')

scores_masked = scores + my_mask
t0.add('+ causal mask', scores_masked, in_shape=scores.shape, note='加性 mask')

attn_weights = torch.softmax(scores_masked, dim=-1, dtype=torch.float32).to(q_rope.dtype)
t0.add('softmax(dim=-1)', attn_weights, in_shape=scores_masked.shape,
       note='每行和为 1')

verify(attn_weights, official.attentions[LAYER], f'L{LAYER} attn_weights')

print()
print(f'每行和是否为 1: {torch.allclose(attn_weights.sum(-1), torch.ones(1, N_HEADS, SEQ))}')
print(f'被 mask 的位置权重最大值: {attn_weights[0, 0, 0, 1:].max().item():.3e}   ← 应为 0')

✓ L0 attn_weights                    max_abs=0.000e+00  mean_abs=0.000e+00  max_rel=0.000e+00

每行和是否为 1: True
被 mask 的位置权重最大值: 0.000e+00   ← 应为 0


In [32]:
# 观察窗口：head 0 的完整 9×9 attention 矩阵
tokens_short = [tokenizer.decode([i]) for i in input_ids[0].tolist()]

def show_attention(weights, head, title):
    print(f'{title}  (行=query 位置，列=key 位置，值 ×100)')
    print(f'{"":>12s}' + ''.join(f'{t:>6s}' for t in tokens_short))
    for i in range(SEQ):
        row = ''.join(
            f'{weights[0, head, i, j].item() * 100:6.1f}' if j <= i else '     ·'
            for j in range(SEQ))
        print(f'{tokens_short[i]:>10s} │{row}')


show_attention(attn_weights, 0, f'Layer {LAYER}, head 0')

Layer 0, head 0  (行=query 位置，列=key 位置，值 ×100)
                北京   是中国     的    首都     ，    巴黎     是    法国     的
        北京 │ 100.0     ·     ·     ·     ·     ·     ·     ·     ·
       是中国 │  44.7  55.3     ·     ·     ·     ·     ·     ·     ·
         的 │   3.8  68.6  27.5     ·     ·     ·     ·     ·     ·
        首都 │  39.0   3.8  13.4  43.8     ·     ·     ·     ·     ·
         ， │   0.7   5.6  13.4   0.2  80.1     ·     ·     ·     ·
        巴黎 │   0.7   2.6  41.6   8.8  37.1   9.2     ·     ·     ·
         是 │   0.4   1.4  35.5   0.2  49.7   1.5  11.3     ·     ·
        法国 │   2.2   2.5  10.3   5.1  20.6  42.7  12.3   4.4     ·
         的 │   1.6  10.6  21.2   0.6  22.5   0.5  16.9   0.4  25.8


### 8.7 加权求和 V，然后 o_proj

$$\text{output} = A V$$

`[B, 16, S, S] @ [B, 16, S, 128] → [B, 16, S, 128]`。第二个 S 被消掉了。

之后要把 16 个头拼回一条向量再过 `o_proj`：

```text
[B, 16, S, 128]
    ↓ transpose(1, 2)      把 S 换回第 1 维
[B, S, 16, 128]
    ↓ contiguous().reshape  16×128 = 2048 拼平
[B, S, 2048]
    ↓ o_proj                2048 → 1024
[B, S, 1024]
```

`transpose` 之后必须 `contiguous()` 才能 `reshape`，因为 transpose 只改了 stride 没搬内存。
这一步的顺序不能反：先 transpose 再 reshape，才能保证同一个位置的 16 个头被拼在一起。

In [33]:
attn_out_heads = torch.matmul(attn_weights, v_rep)
t0.add('A @ V', attn_out_heads, in_shape=attn_weights.shape, note='消掉 key 维')

attn_out_merged = attn_out_heads.transpose(1, 2).contiguous().reshape(BATCH, SEQ, -1)
t0.add('transpose + reshape', attn_out_merged, in_shape=attn_out_heads.shape,
       note='16 头拼成 2048')

attn_output = my_linear(attn_out_merged, attn.o_proj.weight)
t0.add('o_proj', attn_output, in_shape=attn_out_merged.shape, note='2048 → 1024')

verify(attn_output, recorder.out[f'layers.{LAYER}.self_attn.o_proj'], f'L{LAYER} o_proj')
# self_attn 模块返回 (attn_output, attn_weights)
verify(attn_output, recorder.out[f'layers.{LAYER}.self_attn'][0], f'L{LAYER} self_attn 输出')

✓ L0 o_proj                          max_abs=0.000e+00  mean_abs=0.000e+00  max_rel=0.000e+00
✓ L0 self_attn 输出                    max_abs=0.000e+00  mean_abs=0.000e+00  max_rel=0.000e+00


True

### 8.8 第一个残差连接

$$h = x + \mathrm{Attention}(\mathrm{RMSNorm}(x))$$

加的是 §8.1 记下来的 `residual_1`，也就是**未经归一化**的 layer 输入。

In [34]:
hidden_after_attn = residual_1 + attn_output
t0.add('residual add ①', hidden_after_attn, in_shape=attn_output.shape,
       note='加未归一化的 layer 输入')

residual_2 = hidden_after_attn

print('残差前后的量级对比（最后位置，L2 范数）：')
print(f'  residual (layer 输入)  {residual_1[0, -1].norm().item():9.4f}')
print(f'  attention 输出         {attn_output[0, -1].norm().item():9.4f}')
print(f'  相加之后               {hidden_after_attn[0, -1].norm().item():9.4f}')
print()
peek(hidden_after_attn, 'after residual ①', position=-1)

残差前后的量级对比（最后位置，L2 范数）：
  residual (layer 输入)     0.8380
  attention 输出            5.5856
  相加之后                  5.5742

after residual ①             (1, 9, 1024)          
                             [0, -1, :6] = [-0.6333, -0.3618, -0.0587, -0.1963, +0.1306, +0.1057]


### 8.9 MLP：SwiGLU

$$\mathrm{MLP}(x) = W_{\text{down}} \big( \mathrm{SiLU}(W_{\text{gate}} x) \odot W_{\text{up}} x \big)$$

三个投影，两条并行的路径：

```text
        x  [B, S, 1024]
        ├──── gate_proj ───→ [B, S, 3072] ──→ SiLU ──┐
        │                                            ⊙  逐元素相乘
        └──── up_proj ─────→ [B, S, 3072] ───────────┘
                                    │
                              down_proj
                                    ▼
                             [B, S, 1024]
```

`gate` 和 `up` 是两个**独立的**权重矩阵，不是同一个矩阵切两半。
激活只作用在 gate 分支上，up 分支保持线性。

In [35]:
h2 = my_rmsnorm(hidden_after_attn, layer.post_attention_layernorm.weight)
t0.add('post_attention_layernorm', h2, in_shape=hidden_after_attn.shape)
verify(h2, recorder.out[f'layers.{LAYER}.post_attention_layernorm'],
       f'L{LAYER} post_attn_norm')

gate = my_linear(h2, mlp.gate_proj.weight)
up = my_linear(h2, mlp.up_proj.weight)
t0.add('gate_proj', gate, in_shape=h2.shape, note='1024 → 3072')
t0.add('up_proj', up, in_shape=h2.shape, note='1024 → 3072')
verify(gate, recorder.out[f'layers.{LAYER}.mlp.gate_proj'], f'L{LAYER} gate_proj')
verify(up, recorder.out[f'layers.{LAYER}.mlp.up_proj'], f'L{LAYER} up_proj')

activated = my_silu(gate)
gated = activated * up
t0.add('SiLU(gate)', activated, in_shape=gate.shape)
t0.add('SiLU(gate) * up', gated, in_shape=up.shape, note='逐元素相乘')

mlp_output = my_linear(gated, mlp.down_proj.weight)
t0.add('down_proj', mlp_output, in_shape=gated.shape, note='3072 → 1024')
verify(mlp_output, recorder.out[f'layers.{LAYER}.mlp.down_proj'], f'L{LAYER} down_proj')
verify(mlp_output, recorder.out[f'layers.{LAYER}.mlp'], f'L{LAYER} mlp 输出')

✓ L0 post_attn_norm                  max_abs=0.000e+00  mean_abs=0.000e+00  max_rel=0.000e+00


✓ L0 gate_proj                       max_abs=0.000e+00  mean_abs=0.000e+00  max_rel=0.000e+00
✓ L0 up_proj                         max_abs=0.000e+00  mean_abs=0.000e+00  max_rel=0.000e+00
✓ L0 down_proj                       max_abs=2.384e-07  mean_abs=1.458e-08  max_rel=7.242e-08
✓ L0 mlp 输出                          max_abs=2.384e-07  mean_abs=1.458e-08  max_rel=7.242e-08


True

In [36]:
print('gate 分支与 up 分支的数值分布（最后位置，3072 维）：')
for name, tensor in [('gate（激活前）', gate), ('SiLU(gate)', activated),
                     ('up', up), ('相乘之后', gated)]:
    values = tensor[0, -1].float()
    print(f'  {name:<14s} min={values.min():+8.3f}  max={values.max():+8.3f}  '
          f'mean={values.mean():+7.4f}  std={values.std():6.4f}')
print()
_negative_ratio = (gate[0, -1] < 0).float().mean().item()
print(f'gate 为负的比例: {_negative_ratio:.1%}  '
      f'← SiLU 把负值压向 0，起到门控作用')

gate 分支与 up 分支的数值分布（最后位置，3072 维）：
  gate（激活前）      min=  -4.120  max=  +2.457  mean=-0.5787  std=0.7180
  SiLU(gate)     min=  -0.278  max=  +2.263  mean=-0.1244  std=0.1570
  up             min=  -2.989  max=  +1.729  mean=-0.0079  std=0.2873
  相乘之后           min=  -3.699  max=  +0.743  mean=-0.0003  std=0.0962

gate 为负的比例: 83.5%  ← SiLU 把负值压向 0，起到门控作用


### 8.10 第二个残差连接，Layer 0 完成

In [37]:
layer_output = residual_2 + mlp_output
t0.add('residual add ②', layer_output, in_shape=mlp_output.shape)
t0.add('layer output', layer_output, note='= hidden_states[1]')

verify(layer_output, recorder.out[f'layers.{LAYER}'], f'L{LAYER} layer 输出 (hook)')
verify(layer_output, official.hidden_states[LAYER + 1], f'L{LAYER} layer 输出 (hidden_states)')

✓ L0 layer 输出 (hook)                 max_abs=2.384e-07  mean_abs=1.465e-08  max_rel=3.891e-08
✓ L0 layer 输出 (hidden_states)        max_abs=2.384e-07  mean_abs=1.465e-08  max_rel=3.891e-08


True

In [38]:
t0.show()

Layer 0 shape trace
──────────────────────────────────────────────────────────────────────
layer input               (1, 9, 1024)   # = hidden_states[0]，即 embedding 输出
input_layernorm           (1, 9, 1024) → (1, 9, 1024)   # RMSNorm，形状不变
q_proj                    (1, 9, 1024) → (1, 9, 2048)   # 16 头 × 128
k_proj                    (1, 9, 1024) → (1, 9, 1024)   # 8 头 × 128
v_proj                    (1, 9, 1024) → (1, 9, 1024)   # 8 头 × 128
q view                    (1, 9, 2048) → (1, 9, 16, 128)   # 2048 拆成 16×128
k view                    (1, 9, 1024) → (1, 9, 8, 128)   # 1024 拆成 8×128
q_norm                    (1, 9, 16, 128) → (1, 9, 16, 128)   # 在 head_dim=128 上归一化
k_norm                    (1, 9, 8, 128) → (1, 9, 8, 128)   # V 不做 norm
q transpose(1,2)          (1, 9, 16, 128) → (1, 16, 9, 128)   # 头维提前
k transpose(1,2)          (1, 9, 8, 128) → (1, 8, 9, 128)
v transpose(1,2)          (1, 9, 8, 128) → (1, 8, 9, 128)
RoPE(q)                   (1, 16, 9, 128) → (1, 16, 9, 128)   # 形

Layer 0 全部 25 个节点逐一对齐。回头看这张 trace，形状变化可以归成三段：

- **升维**：1024 → 2048（q_proj）或 1024 → 3072（gate/up_proj）；
- **序列维出现两次**：`[B, 16, S, S]` 是唯一一处形状与序列长度成平方关系的张量；
- **降回 1024**：o_proj 和 down_proj 把宽度还原，残差才能相加。

整个 Layer 是保形的：进去 `[1, 9, 1024]`，出来 `[1, 9, 1024]`。

## 9. 封装成模块，跑完 28 层

把上面验证过的函数组装成与 Qwen3 源码结构对应的类：

```text
MyQwen3ForCausalLM
└── MyQwen3Model
    ├── my_embedding
    ├── MyQwen3DecoderLayer × 28
    │   ├── MyQwen3RMSNorm      (input_layernorm)
    │   ├── MyQwen3Attention
    │   │   ├── MyQwen3RMSNorm  (q_norm / k_norm)
    │   │   └── RoPE / GQA / causal attention
    │   ├── MyQwen3RMSNorm      (post_attention_layernorm)
    │   └── MyQwen3MLP
    ├── MyQwen3RMSNorm          (final norm)
    └── lm_head（与 embedding 共享权重）
```

参数全部通过引用真实模型的 `weight` 张量获得，**不复制、不重新初始化**。

In [39]:
class MyQwen3RMSNorm:
    """对应 Qwen3RMSNorm。"""

    def __init__(self, weight, eps=RMS_EPS):
        self.weight = weight
        self.eps = eps

    def __call__(self, x):
        return my_rmsnorm(x, self.weight, self.eps)


class MyQwen3MLP:
    """对应 Qwen3MLP，SwiGLU 结构。"""

    def __init__(self, source_mlp):
        self.gate_weight = source_mlp.gate_proj.weight
        self.up_weight = source_mlp.up_proj.weight
        self.down_weight = source_mlp.down_proj.weight

    def __call__(self, x):
        gate = my_linear(x, self.gate_weight)
        up = my_linear(x, self.up_weight)
        return my_linear(my_silu(gate) * up, self.down_weight)

In [40]:
class MyQwen3Attention:
    """对应 Qwen3Attention + eager_attention_forward。"""

    def __init__(self, source_attn):
        self.q_weight = source_attn.q_proj.weight
        self.k_weight = source_attn.k_proj.weight
        self.v_weight = source_attn.v_proj.weight
        self.o_weight = source_attn.o_proj.weight
        self.q_norm = MyQwen3RMSNorm(source_attn.q_norm.weight)
        self.k_norm = MyQwen3RMSNorm(source_attn.k_norm.weight)

    def __call__(self, x, cos, sin, mask, collect=None):
        batch, seq_len, _ = x.shape
        head_shape = (batch, seq_len, -1, HEAD_DIM)

        # 投影 → 拆头 → 逐头 norm → 头维提前
        q = self.q_norm(my_linear(x, self.q_weight).view(head_shape)).transpose(1, 2)
        k = self.k_norm(my_linear(x, self.k_weight).view(head_shape)).transpose(1, 2)
        v = my_linear(x, self.v_weight).view(head_shape).transpose(1, 2)

        q, k = my_apply_rope(q, k, cos, sin)          # RoPE 在 norm 之后
        k = my_repeat_kv(k, N_REP)                    # GQA 在 RoPE 之后
        v = my_repeat_kv(v, N_REP)

        scores = torch.matmul(q, k.transpose(2, 3)) * SCALING
        if mask is not None:
            scores = scores + mask
        weights = torch.softmax(scores, dim=-1, dtype=torch.float32).to(q.dtype)

        out = torch.matmul(weights, v).transpose(1, 2).contiguous()
        out = out.reshape(batch, seq_len, -1)
        if collect is not None:
            collect['attn_weights'] = weights
        return my_linear(out, self.o_weight)

In [41]:
class MyQwen3DecoderLayer:
    """对应 Qwen3DecoderLayer。pre-norm + 两次残差。"""

    def __init__(self, source_layer):
        self.input_layernorm = MyQwen3RMSNorm(source_layer.input_layernorm.weight)
        self.self_attn = MyQwen3Attention(source_layer.self_attn)
        self.post_attention_layernorm = MyQwen3RMSNorm(
            source_layer.post_attention_layernorm.weight)
        self.mlp = MyQwen3MLP(source_layer.mlp)

    def __call__(self, hidden_states, cos, sin, mask, collect=None):
        residual = hidden_states
        hidden_states = self.input_layernorm(hidden_states)
        hidden_states = self.self_attn(hidden_states, cos, sin, mask, collect)
        hidden_states = residual + hidden_states

        residual = hidden_states
        hidden_states = self.post_attention_layernorm(hidden_states)
        hidden_states = self.mlp(hidden_states)
        return residual + hidden_states


class MyQwen3Model:
    """对应 Qwen3Model：embedding + 28 层 + final norm。"""

    def __init__(self, source_model):
        self.embed_weight = source_model.embed_tokens.weight
        self.layers = [MyQwen3DecoderLayer(layer) for layer in source_model.layers]
        self.norm = MyQwen3RMSNorm(source_model.norm.weight)

    def __call__(self, ids, collect_attn=False):
        seq_len = ids.shape[1]
        position_ids = torch.arange(seq_len, device=ids.device).unsqueeze(0)
        cos, sin, _ = my_rope_tables(position_ids)
        mask = my_causal_mask(seq_len)

        hidden = my_embedding(ids, self.embed_weight)
        all_hidden = [hidden]
        all_attn = []
        for layer in self.layers:
            bucket = {} if collect_attn else None
            hidden = layer(hidden, cos, sin, mask, bucket)
            all_hidden.append(hidden)
            if collect_attn:
                all_attn.append(bucket['attn_weights'])
        return self.norm(hidden), all_hidden, all_attn

In [42]:
class MyQwen3ForCausalLM:
    """对应 Qwen3ForCausalLM。lm_head 与 embedding 共享权重。"""

    def __init__(self, source):
        self.model = MyQwen3Model(source.model)
        # 权重绑定：直接引用同一个张量，不复制
        self.lm_head_weight = source.model.embed_tokens.weight

    def __call__(self, ids, collect_attn=False):
        last_hidden, all_hidden, all_attn = self.model(ids, collect_attn)
        logits = my_linear(last_hidden, self.lm_head_weight)
        return logits, last_hidden, all_hidden, all_attn


my_model = MyQwen3ForCausalLM(model)
print(f'复现模型层数: {len(my_model.model.layers)}')
print(f'lm_head 权重与 embedding 是同一对象: '
      f'{my_model.lm_head_weight is my_model.model.embed_weight}')
print(f'与官方模型共享权重（未复制）: '
      f'{my_model.model.layers[0].mlp.gate_weight is model.model.layers[0].mlp.gate_proj.weight}')

复现模型层数: 28
lm_head 权重与 embedding 是同一对象: True
与官方模型共享权重（未复制）: True


### 9.1 跑完整个模型，逐层验证

关键设计：**每层都用自己上一层的输出作为输入**，不从官方结果里"借"中间态。
这样误差会累积，才能真正检验复现的正确性。如果只用官方的 `hidden_states[i]` 当第 i 层输入，
每层误差都会被重置，验证的强度大打折扣。

在比对之前必须先确认一件事：`output_hidden_states` 返回的 29 个张量，最后一个到底是什么。

第一次写这段验证时我假设 `hidden_states[28]` 就是 Layer 27 的输出，结果那一层报出 460 的巨大误差，
而 attention 权重完全正确——这种"只有最后一层错、且错得离谱"的形态说明不是数值累积，是语义理解错了。
下面用 hook 直接查证。

In [43]:
_l27_hook = recorder.out[f'layers.{N_LAYERS - 1}']
_norm_hook = recorder.out['final_norm']
_hs28 = official.hidden_states[N_LAYERS]

print(f'hidden_states[{N_LAYERS}] == Layer 27 的 hook 输出 : '
      f'{torch.equal(_hs28, _l27_hook)}')
print(f'hidden_states[{N_LAYERS}] == final_norm 的输出    : '
      f'{torch.equal(_hs28, _norm_hook)}')
print()
print(f'L2 范数对比:')
print(f'  Layer 27 输出（norm 前） {_l27_hook.norm().item():10.3f}')
print(f'  final_norm 输出          {_norm_hook.norm().item():10.3f}')
print(f'  hidden_states[28]        {_hs28.norm().item():10.3f}')

hidden_states[28] == Layer 27 的 hook 输出 : False
hidden_states[28] == final_norm 的输出    : True

L2 范数对比:
  Layer 27 输出（norm 前）   1510.185
  final_norm 输出             391.355
  hidden_states[28]           391.355


**`hidden_states` 的下标语义不是均匀的：**

| 下标 | 含义 |
|---|---|
| `hidden_states[0]` | embedding 输出（= Layer 0 的输入） |
| `hidden_states[i]`，1 ≤ i ≤ 27 | Layer i-1 的输出（= Layer i 的输入） |
| `hidden_states[28]` | **final RMSNorm 之后**的结果，不是 Layer 27 的原始输出 |

所以验证 Layer 27 时要拿 hook 抓到的输出比，而不是 `hidden_states[28]`。

In [44]:
my_logits, my_last_hidden, my_all_hidden, my_all_attn = my_model(
    input_ids, collect_attn=True)

print(f'my_logits        {tuple(my_logits.shape)}')
print(f'my_all_hidden    {len(my_all_hidden)} 个 × {tuple(my_all_hidden[0].shape)}')
print(f'my_all_attn      {len(my_all_attn)} 个 × {tuple(my_all_attn[0].shape)}')

my_logits        (1, 9, 151936)
my_all_hidden    29 个 × (1, 9, 1024)
my_all_attn      28 个 × (1, 16, 9, 9)


判定标准也需要说明。hidden state 里存在量级 6000 以上的元素（§9.2 会追查这件事），
对这种张量用固定的绝对公差没有意义。这里用**尺度相对误差**：

$$\text{rel} = \frac{\max |{\rm mine} - {\rm ref}|}{\max |{\rm ref}|}$$

float32 的机器精度是 `1.19e-07`。经过 28 层累积，达到 1e-6 量级属于正常范围，
判定阈值取 `1e-5`。

In [45]:
# 逐层对齐：hidden state 与 attention 权重
REL_THRESHOLD = 1e-5

print(f'{"层":>3s}  {"hidden max_abs":>14s}  {"hidden mean_abs":>15s}  '
      f'{"scale_rel":>11s}  {"attn max_abs":>13s}  ok')
print('─' * 74)

layer_errors = []
first_failure = None

for layer_index in range(N_LAYERS):
    mine = my_all_hidden[layer_index + 1]
    # 用 hook 抓到的每层真实输出作为基准。最后一层不能用 hidden_states[28]，
    # 因为那已经过了 final norm（见上面的查证）。
    ref = recorder.out[f'layers.{layer_index}']
    diff = (mine.float() - ref.float()).abs()
    max_abs = diff.max().item()
    mean_abs = diff.mean().item()
    scale_rel = max_abs / ref.float().abs().max().item()

    attn_diff = (my_all_attn[layer_index].float()
                 - official.attentions[layer_index].float()).abs().max().item()

    ok = scale_rel < REL_THRESHOLD and attn_diff < 1e-5
    if not ok and first_failure is None:
        first_failure = layer_index
    layer_errors.append((layer_index, max_abs, mean_abs, scale_rel, attn_diff, ok))

    flag = '✓' if ok else '✗'
    print(f'{layer_index:>3d}  {max_abs:>14.3e}  {mean_abs:>15.3e}  '
          f'{scale_rel:>11.3e}  {attn_diff:>13.3e}  {flag}')

print('─' * 74)
_all_ok = all(row[5] for row in layer_errors)
print(f'全部 {N_LAYERS} 层通过（scale_rel < {REL_THRESHOLD:.0e}）: {_all_ok}')
if first_failure is not None:
    print(f'⚠ 首个不一致的层: Layer {first_failure}')
else:
    print('未出现不一致，无需定位首个误差节点。')

VERIFICATIONS.append((f'全部 {N_LAYERS} 层 hidden state', _all_ok,
                      max(row[3] for row in layer_errors)))
VERIFICATIONS.append((f'全部 {N_LAYERS} 层 attention 权重',
                      all(row[4] < 1e-5 for row in layer_errors),
                      max(row[4] for row in layer_errors)))

  层  hidden max_abs  hidden mean_abs    scale_rel   attn max_abs  ok
──────────────────────────────────────────────────────────────────────────
  0       2.384e-07        1.465e-08    3.891e-08      0.000e+00  ✓
  1       1.192e-06        6.816e-08    1.344e-07      8.643e-07  ✓
  2       1.465e-03        3.720e-07    2.291e-07      1.252e-06  ✓
  3       1.465e-03        4.077e-07    2.292e-07      7.153e-07  ✓
  4       1.465e-03        4.383e-07    2.292e-07      8.941e-07  ✓
  5       1.465e-03        4.893e-07    2.292e-07      7.153e-07  ✓
  6       1.465e-03        5.390e-07    2.293e-07      1.341e-06  ✓
  7       1.465e-03        5.955e-07    2.293e-07      9.537e-07  ✓
  8       1.465e-03        6.520e-07    2.295e-07      1.192e-06  ✓
  9       1.465e-03        7.016e-07    2.297e-07      1.192e-06  ✓
 10       1.465e-03        7.973e-07    2.297e-07      9.835e-07  ✓
 11       1.465e-03        8.944e-07    2.298e-07      1.073e-06  ✓
 12       1.465e-03        9.582e-07    

### 9.2 那个恒定不变的 max_abs 是什么

上表里 `hidden max_abs` 从第 2 层起就锁定在 `1.465e-03` 不再变化，而 `mean_abs` 一直在缓慢增长。
最大误差不随层数增长，说明它不是累积效应，而是**某个特定元素的浮点精度极限**。

In [46]:
_probe = official.hidden_states[10]
print(f'hidden_states[10] 的绝对值分布: max={_probe.abs().max().item():.1f}  '
      f'mean={_probe.abs().mean().item():.4f}')
print()
print('最大的 5 个元素：')
_top = _probe.abs().flatten().topk(5)
for value, flat_index in zip(_top.values, _top.indices):
    seq_pos = (flat_index // HIDDEN % SEQ).item()
    channel = (flat_index % HIDDEN).item()
    print(f'  |{value.item():9.1f}|  在 [0, {seq_pos}, {channel}]')
print()
_ulp = torch.finfo(torch.float32).eps * _probe.abs().max().item()
print(f'float32 在 |x|≈{_probe.abs().max().item():.0f} 处的最小间隔（1 ULP）= {_ulp:.3e}')
print(f'观察到的 max_abs = 1.465e-03 ≈ {1.465e-3 / _ulp:.1f} ULP')
print()
print('结论：最大误差来自数值最大的那个元素，是 float32 表示精度的下限，不是实现错误。')

hidden_states[10] 的绝对值分布: max=6378.2  mean=1.6934

最大的 5 个元素：
  |   6378.2|  在 [0, 0, 35]
  |    628.1|  在 [0, 0, 13]
  |    364.1|  在 [0, 0, 1]
  |    154.3|  在 [0, 0, 277]
  |    106.9|  在 [0, 0, 7]

float32 在 |x|≈6378 处的最小间隔（1 ULP）= 7.603e-04
观察到的 max_abs = 1.465e-03 ≈ 1.9 ULP

结论：最大误差来自数值最大的那个元素，是 float32 表示精度的下限，不是实现错误。


这里浮出一个意料之外的现象：**位置 0 的第 35 号通道，数值高达 6378，比全张量均值大了三个数量级。**
这不是我们预设要观察的东西，是验证过程中撞见的。放到 §11 一起看它沿层的演化。

## 10. 出口：final norm → LM Head → Top-K → token

```text
   Layer 27 输出  [B, S, 1024]
         ↓
   final RMSNorm                    model.model.norm
         ↓        [B, S, 1024]
         ↓
   lm_head        ← 复用 embedding 那张 [151936, 1024] 表
         ↓        [B, S, 151936]
         ↓
   取最后一个位置  [151936]
         ↓
   Top-K
         ↓
   token id → decode
```

LM Head 的计算就是 `hidden @ embed_weight.T`：把 1024 维的 hidden state 与词表里 151936 个
token 向量逐个做内积。**内积大 = 方向接近 = 得分高。**
入口那次查表和出口这次打分，用的是同一张矩阵。

In [47]:
my_final_hidden = my_model.model.norm(my_all_hidden[-1])
verify(my_final_hidden, recorder.out['final_norm'], 'final RMSNorm', atol=1e-3, rtol=1e-4)
verify(my_final_hidden, official.hidden_states[N_LAYERS], 'final norm vs hidden_states[28]',
       atol=1e-3, rtol=1e-4)

verify(my_logits, official.logits, 'lm_head logits', atol=1e-2, rtol=1e-3)

print()
print(f'官方 logits 数值范围: [{official.logits.min().item():.3f}, '
      f'{official.logits.max().item():.3f}]')
print(f'复现 logits 数值范围: [{my_logits.min().item():.3f}, {my_logits.max().item():.3f}]')

✓ final RMSNorm                      max_abs=8.297e-05  mean_abs=2.408e-06  max_rel=1.051e-06
✓ final norm vs hidden_states[28]    max_abs=8.297e-05  mean_abs=2.408e-06  max_rel=1.051e-06
✓ lm_head logits                     max_abs=3.242e-05  mean_abs=3.097e-06  max_rel=1.373e-06

官方 logits 数值范围: [-18.614, 23.609]
复现 logits 数值范围: [-18.614, 23.609]


logits 的公差比前面各节点宽（`atol=1e-2`）。原因是 §9.2 那个量级 6000+ 的元素经过
final norm 和一次 1024→151936 的矩阵乘后，误差被同比例放大。
判断标准应该看**相对误差**和**预测是否一致**，下面直接验证后者。

In [48]:
TOP_K = 10
last_position = SEQ - 1

official_last = official.logits[0, last_position]
my_last = my_logits[0, last_position]

official_top = official_last.topk(TOP_K)
my_top = my_last.topk(TOP_K)

official_probs = torch.softmax(official_last, dim=-1)

print(f'输入: {PROMPT}')
print(f'预测第 {SEQ + 1} 个 token（基于位置 {last_position} 的 logits）\n')
print(f'{"排名":>4s}  {"官方 token":<12s} {"logit":>8s} {"概率":>8s}   '
      f'{"复现 token":<12s} {"logit":>8s}  一致')
print('─' * 74)
for rank in range(TOP_K):
    o_id = official_top.indices[rank].item()
    m_id = my_top.indices[rank].item()
    o_text = tokenizer.decode([o_id])
    m_text = tokenizer.decode([m_id])
    prob = official_probs[o_id].item()
    same = '✓' if o_id == m_id else '✗'
    print(f'{rank + 1:>4d}  {o_text!r:<12s} {official_top.values[rank].item():>8.3f} '
          f'{prob:>7.2%}   {m_text!r:<12s} {my_top.values[rank].item():>8.3f}  {same}')

输入: 北京是中国的首都，巴黎是法国的
预测第 10 个 token（基于位置 8 的 logits）

  排名  官方 token        logit       概率   复现 token        logit  一致
──────────────────────────────────────────────────────────────────────────
   1  '首都'           23.249  95.94%   '首都'           23.249  ✓
   2  '首'            19.600   2.50%   '首'            19.600  ✓
   3  '象征'           17.285   0.25%   '象征'           17.285  ✓
   4  '都'            17.098   0.20%   '都'            17.098  ✓
   5  '国家'           16.898   0.17%   '国家'           16.898  ✓
   6  '____'         15.765   0.05%   '____'         15.765  ✓
   7  '代表'           15.759   0.05%   '代表'           15.759  ✓
   8  '代'            15.716   0.05%   '代'            15.716  ✓
   9  '省'            15.660   0.05%   '省'            15.660  ✓
  10  '第二'           15.353   0.04%   '第二'           15.353  ✓


In [49]:
# 闭环验证：官方与复现的 argmax 预测是否一致（全部 9 个位置）
official_argmax = official.logits[0].argmax(-1)
my_argmax = my_logits[0].argmax(-1)
all_match = torch.equal(official_argmax, my_argmax)

print(f'{"位置":>4s}  {"输入 token":<10s} → {"官方预测":<10s} {"复现预测":<10s} 一致')
print('─' * 52)
for position in range(SEQ):
    source = tokenizer.decode([input_ids[0, position].item()])
    o_pred = tokenizer.decode([official_argmax[position].item()])
    m_pred = tokenizer.decode([my_argmax[position].item()])
    flag = '✓' if official_argmax[position] == my_argmax[position] else '✗'
    print(f'{position:>4d}  {source!r:<10s} → {o_pred!r:<10s} {m_pred!r:<10s}  {flag}')
print('─' * 52)
print(f'全部 {SEQ} 个位置预测一致: {all_match}')

VERIFICATIONS.append(('Top-K 预测一致（全部位置）', all_match, 0.0))

  位置  输入 token   → 官方预测       复现预测       一致
────────────────────────────────────────────────────
   0  '北京'       → '地铁'       '地铁'        ✓
   1  '是中国'      → '的'        '的'         ✓
   2  '的'        → '首都'       '首都'        ✓
   3  '首都'       → '，'        '，'         ✓
   4  '，'        → '也是'       '也是'        ✓
   5  '巴黎'       → '是'        '是'         ✓
   6  '是'        → '法国'       '法国'        ✓
   7  '法国'       → '的'        '的'         ✓
   8  '的'        → '首都'       '首都'        ✓
────────────────────────────────────────────────────
全部 9 个位置预测一致: True


每个位置都在预测"它的下一个 token"，这是 causal 语言模型的定义。位置 8（最后一个 `的`）
的预测才是我们关心的续写结果。前面几个位置的预测顺带展示了模型在读到一半时的判断。

In [50]:
# 把预测接回原文，完成一次完整闭环
next_token_id = official_argmax[last_position].item()
continuation = tokenizer.decode([next_token_id])

print(f'原文:   {PROMPT}')
print(f'续写:   {PROMPT}{continuation}')
print()
print(f'预测 token id = {next_token_id}, 文本 = {continuation!r}')
print(f'概率 = {official_probs[next_token_id].item():.2%}')
print()
_second = official_top.indices[1].item()
print(f'与第二名的 logit 差距: '
      f'{official_top.values[0].item() - official_top.values[1].item():.3f}')
print(f'第二名: {tokenizer.decode([_second])!r} '
      f'({official_probs[_second].item():.2%})')

原文:   北京是中国的首都，巴黎是法国的
续写:   北京是中国的首都，巴黎是法国的首都

预测 token id = 106114, 文本 = '首都'
概率 = 95.94%

与第二名的 logit 差距: 3.649
第二名: '首' (2.50%)


## 11. 让数据决定看哪几层

28 层结构相同，但行为不一定相同。这一节先算出每层的统计量，**再**根据数据挑代表层，
而不是预先指定"看第 0、13、27 层"。

统计量选这几个，都能从已有记录直接算：

- **hidden 范数**：这一层输出的量级；
- **相对变化**：`‖layer_out − layer_in‖ / ‖layer_in‖`，衡量这层改动了多少；
- **attention 熵**：注意力分布的集中程度，低熵表示聚焦在少数位置；
- **对角占比**：attention 权重落在"看自己"位置上的比例；
- **首位占比**：attention 权重落在位置 0 上的比例。

In [51]:
import math

stats = []
for layer_index in range(N_LAYERS):
    layer_in = official.hidden_states[layer_index]
    layer_out = recorder.out[f'layers.{layer_index}']
    weights = official.attentions[layer_index][0].float()   # [heads, S, S]

    norm_out = layer_out.float().norm().item()
    delta = (layer_out.float() - layer_in.float()).norm().item()
    rel_change = delta / layer_in.float().norm().item()

    # 只统计有效（未被 mask）的行，逐 query 位置算熵后取平均
    entropies = []
    for position in range(SEQ):
        row = weights[:, position, :position + 1]
        entropy = -(row * (row + 1e-12).log()).sum(-1).mean().item()
        max_entropy = math.log(position + 1) if position > 0 else 1.0
        entropies.append(entropy / max_entropy if position > 0 else 0.0)
    mean_entropy = sum(entropies[1:]) / (SEQ - 1)

    diagonal = weights.diagonal(dim1=-2, dim2=-1).mean().item()
    first_column = weights[:, 1:, 0].mean().item()
    max_activation = layer_out.abs().max().item()

    stats.append(dict(layer=layer_index, norm=norm_out, rel_change=rel_change,
                      entropy=mean_entropy, diagonal=diagonal,
                      first_col=first_column, max_act=max_activation))

print(f'{"层":>3s} {"‖out‖":>10s} {"相对变化":>9s} {"归一熵":>8s} '
      f'{"对角占比":>9s} {"首位占比":>9s} {"max|act|":>10s}')
print('─' * 66)
for row in stats:
    print(f'{row["layer"]:>3d} {row["norm"]:>10.1f} {row["rel_change"]:>9.4f} '
          f'{row["entropy"]:>8.4f} {row["diagonal"]:>9.4f} '
          f'{row["first_col"]:>9.4f} {row["max_act"]:>10.1f}')

  层      ‖out‖      相对变化      归一熵      对角占比      首位占比   max|act|
──────────────────────────────────────────────────────────────────
  0       32.7   12.3651   0.4365    0.6329    0.0622        6.1
  1       41.1    0.5952   0.6740    0.4032    0.1182        8.9
  2     6440.8  156.5395   0.5849    0.4132    0.1420     6392.5
  3     6440.2    0.0029   0.3287    0.1638    0.8101     6391.8
  4     6439.8    0.0032   0.4260    0.2252    0.7306     6391.4
  5     6439.9    0.0040   0.4225    0.2108    0.7179     6391.3
  6     6438.1    0.0040   0.4446    0.2269    0.6320     6389.5
  7     6436.4    0.0045   0.5690    0.2408    0.6409     6387.7
  8     6432.8    0.0047   0.4899    0.2165    0.6631     6383.8
  9     6427.4    0.0053   0.5019    0.2557    0.6511     6378.2
 10     6426.6    0.0065   0.6386    0.2430    0.5758     6377.0
 11     6423.6    0.0070   0.4377    0.2817    0.6154     6373.5
 12     6421.5    0.0058   0.6384    0.1982    0.5826     6371.1
 13     6419.4    0.005

### 11.1 按数据挑代表层

上表里有几处数字明显偏离邻居。用几条客观判据把它们选出来。

In [52]:
def pick(key, mode='max', exclude=()):
    candidates = [r for r in stats if r['layer'] not in exclude]
    chosen = (max if mode == 'max' else min)(candidates, key=lambda r: r[key])
    return chosen['layer']


CRITERIA = [
    ('相对变化最大',   pick('rel_change', 'max'),          'rel_change'),
    ('输出范数最大',   pick('norm', 'max'),                'norm'),
    ('注意力最聚焦',   pick('entropy', 'min', exclude=(0,)), 'entropy'),
    ('首位占比最高',   pick('first_col', 'max'),           'first_col'),
    ('对角占比最高',   pick('diagonal', 'max'),            'diagonal'),
    ('最后一层',       N_LAYERS - 1,                       'rel_change'),
]

print(f'{"判据":<14s} {"层":>4s}  {"该指标数值":>12s}')
print('─' * 36)
for name, layer_index, key in CRITERIA:
    print(f'{name:<14s} {layer_index:>4d}  {stats[layer_index][key]:>12.4f}')

REPRESENTATIVE = sorted({layer_index for _, layer_index, _ in CRITERIA})
print()
print(f'代表层 = {REPRESENTATIVE}')

判据                层         该指标数值
────────────────────────────────────
相对变化最大            2      156.5395
输出范数最大           25     6634.9897
注意力最聚焦           25        0.2310
首位占比最高           24        0.8940
对角占比最高            0        0.6329
最后一层             27        0.9042

代表层 = [0, 2, 24, 25, 27]


In [53]:
# 相对变化的分布：哪些层几乎什么都没做，哪些层改动巨大
_sorted = sorted(stats, key=lambda r: r['rel_change'], reverse=True)
print('相对变化排名（前 5 / 后 5）：')
for row in _sorted[:5]:
    print(f'  Layer {row["layer"]:>2d}  rel_change = {row["rel_change"]:>10.4f}')
print('  ...')
for row in _sorted[-5:]:
    print(f'  Layer {row["layer"]:>2d}  rel_change = {row["rel_change"]:>10.4f}')
print()
_middle = [r['rel_change'] for r in stats if 3 <= r['layer'] <= 25]
print(f'Layer 3–25 的相对变化: 最小 {min(_middle):.4f}  最大 {max(_middle):.4f}  '
      f'中位数 {sorted(_middle)[len(_middle) // 2]:.4f}')
print('这些层对残差流的相对改动都在百分之几以内。')

相对变化排名（前 5 / 后 5）：
  Layer  2  rel_change =   156.5395
  Layer  0  rel_change =    12.3651
  Layer 27  rel_change =     0.9042
  Layer  1  rel_change =     0.5952
  Layer 26  rel_change =     0.0898
  ...
  Layer  7  rel_change =     0.0045
  Layer  6  rel_change =     0.0040
  Layer  5  rel_change =     0.0040
  Layer  4  rel_change =     0.0032
  Layer  3  rel_change =     0.0029

Layer 3–25 的相对变化: 最小 0.0029  最大 0.0534  中位数 0.0079
这些层对残差流的相对改动都在百分之几以内。


### 11.2 追踪那个巨大的激活值

§9.2 撞见位置 0 的第 35 号通道数值达到 6378。结合 §11 的表可以看出，
`max|act|` 在 Layer 2 从 8.9 跳到 6392.5，然后一路保持到 Layer 25，最后两层又被压下去。
下面把这个通道沿 29 个 hidden state 逐层打印出来。

In [54]:
# 先确认每层最大激活出现的位置和通道是否稳定
print(f'{"层":>3s}  {"max|act|":>10s}  {"位置":>4s}  {"通道":>5s}   该层 |act| 均值')
print('─' * 52)
outlier_track = []
for layer_index in range(N_LAYERS + 1):
    tensor = official.hidden_states[layer_index][0].float()   # [S, H]
    flat_index = tensor.abs().argmax().item()
    position, channel = flat_index // HIDDEN, flat_index % HIDDEN
    value = tensor[position, channel].item()
    outlier_track.append((layer_index, position, channel, value))
    if layer_index <= 4 or layer_index >= N_LAYERS - 3 or layer_index % 6 == 0:
        print(f'{layer_index:>3d}  {value:>10.1f}  {position:>4d}  {channel:>5d}   '
              f'{tensor.abs().mean().item():>8.4f}')

_channels = {c for _, _, c, _ in outlier_track[3:]}
_positions = {p for _, p, _, _ in outlier_track[3:]}
print()
print(f'从 Layer 2 输出起，最大激活所在通道集合 = {_channels}')
print(f'                    所在位置集合 = {_positions}')
print()
print('出现了两个不同的离群点，不是一个。分别锁定：')
print('  A: Layer 2 起长期占据榜首的那个')
print('  B: final norm 之后仍居榜首的那个')

  层    max|act|    位置     通道   该层 |act| 均值
────────────────────────────────────────────────────
  0         0.1     1    297     0.0224
  1         6.1     4     35     0.2175
  2         8.9     1     35     0.2450
  3      6392.5     0     35     1.3966
  4      6391.8     0     35     1.4287
  6      6391.3     0     35     1.5209
 12      6373.5     0     35     1.8895
 18      6362.4     0     35     2.8392
 24      6402.2     0     35     7.0008
 25      6406.4     0     35     7.5205
 26      6406.5     0     35     8.1453
 27      6097.4     0     35     8.4598
 28        78.9     2     27     2.2280

从 Layer 2 输出起，最大激活所在通道集合 = {27, 35}
                    所在位置集合 = {0, 2}

出现了两个不同的离群点，不是一个。分别锁定：
  A: Layer 2 起长期占据榜首的那个
  B: final norm 之后仍居榜首的那个


In [55]:
# A 取网络中段的榜首，B 取 final norm 之后的榜首
_, POS_A, CH_A, _ = outlier_track[15]
_, POS_B, CH_B, _ = outlier_track[N_LAYERS]

print(f'A = 位置 {POS_A}, 通道 {CH_A}')
print(f'B = 位置 {POS_B}, 通道 {CH_B}')
print()
print(f'{"index":>6s}  {"含义":<20s}  {"A 数值":>10s}  {"B 数值":>10s}')
print('─' * 54)
for layer_index in range(N_LAYERS + 1):
    tensor = official.hidden_states[layer_index][0]
    value_a = tensor[POS_A, CH_A].item()
    value_b = tensor[POS_B, CH_B].item()
    if layer_index == 0:
        meaning = 'embedding 输出'
    elif layer_index == N_LAYERS:
        meaning = 'final norm 之后'
    else:
        meaning = f'Layer {layer_index - 1} 输出'
    if layer_index <= 3 or layer_index >= N_LAYERS - 2 or layer_index % 5 == 0:
        print(f'{layer_index:>6d}  {meaning:<20s}  {value_a:>10.1f}  {value_b:>10.1f}')
print()
print(f'同位置其他通道的中位量级: A 处 '
      f'{official.hidden_states[15][0, POS_A].abs().median().item():.4f}，'
      f'B 处 {official.hidden_states[15][0, POS_B].abs().median().item():.4f}')

A = 位置 0, 通道 35
B = 位置 2, 通道 27

 index  含义                          A 数值        B 数值
──────────────────────────────────────────────────────
     0  embedding 输出                -0.0         0.0
     1  Layer 0 输出                   3.1        -0.9
     2  Layer 1 输出                   1.6        -0.1
     3  Layer 2 输出                6392.5        -0.0
     5  Layer 4 输出                6391.4         0.2
    10  Layer 9 输出                6378.2        -0.0
    15  Layer 14 输出               6364.7         3.9
    20  Layer 19 输出               6375.9        23.4
    25  Layer 24 输出               6406.4        78.5
    26  Layer 25 输出               6406.5        85.8
    27  Layer 26 输出               6097.4        99.3
    28  final norm 之后               -2.1        78.9

同位置其他通道的中位量级: A 处 1.5053，B 处 0.5960


两个离群点的形态完全不同：

- **A（位置 0，通道 35）**：在 **Layer 2 内部**被一次性写入约 6390，之后 23 层几乎原样保留，
  相对改动只有百分之几。它出现在序列的第一个 token 上。
- **B（位置 2，通道 27）**：从中段开始**逐层累积**，到 Layer 26 输出时约 99。
  它是 final norm 之后仍然最大的元素。

A 解释了 §9.2 那个恒定不变的 `max_abs`：它的量级决定了 float32 在该张量上的精度下限。

结合 §11 表里"首位占比"从 Layer 3 起长期维持在 0.55–0.89，这两个现象是同一件事的两面：
大量注意力头把权重压在位置 0 上。至于机制层面的解释，超出本实验"只弄清怎么算"的范围，
留作观察记录。

### 11.3 代表层的 attention 矩阵

同一个 head 编号在不同层的行为差别很大。挑 §11.1 选出的代表层，各看一个 head。

In [56]:
for layer_index in REPRESENTATIVE:
    reasons = [name for name, index, _ in CRITERIA if index == layer_index]
    print(f'\n{"=" * 70}')
    print(f'Layer {layer_index}   入选判据: {", ".join(reasons)}')
    print(f'  归一熵={stats[layer_index]["entropy"]:.4f}  '
          f'对角={stats[layer_index]["diagonal"]:.4f}  '
          f'首位={stats[layer_index]["first_col"]:.4f}')
    show_attention(official.attentions[layer_index], 0, f'  head 0')


Layer 0   入选判据: 对角占比最高
  归一熵=0.4365  对角=0.6329  首位=0.0622
  head 0  (行=query 位置，列=key 位置，值 ×100)
                北京   是中国     的    首都     ，    巴黎     是    法国     的
        北京 │ 100.0     ·     ·     ·     ·     ·     ·     ·     ·
       是中国 │  44.7  55.3     ·     ·     ·     ·     ·     ·     ·
         的 │   3.8  68.6  27.5     ·     ·     ·     ·     ·     ·
        首都 │  39.0   3.8  13.4  43.8     ·     ·     ·     ·     ·
         ， │   0.7   5.6  13.4   0.2  80.1     ·     ·     ·     ·
        巴黎 │   0.7   2.6  41.6   8.8  37.1   9.2     ·     ·     ·
         是 │   0.4   1.4  35.5   0.2  49.7   1.5  11.3     ·     ·
        法国 │   2.2   2.5  10.3   5.1  20.6  42.7  12.3   4.4     ·
         的 │   1.6  10.6  21.2   0.6  22.5   0.5  16.9   0.4  25.8

Layer 2   入选判据: 相对变化最大
  归一熵=0.5849  对角=0.4132  首位=0.1420
  head 0  (行=query 位置，列=key 位置，值 ×100)
                北京   是中国     的    首都     ，    巴黎     是    法国     的
        北京 │ 100.0     ·     ·     ·     ·     ·     ·     ·     ·


In [57]:
# 同一层内不同 head 的差异：用首位占比排序，看最极端的两个
LOOK_AT = REPRESENTATIVE[len(REPRESENTATIVE) // 2]
weights = official.attentions[LOOK_AT][0].float()
per_head_first = weights[:, 1:, 0].mean(dim=(1,))

print(f'Layer {LOOK_AT} 各 head 的首位占比：')
for head in range(N_HEADS):
    bar = '█' * int(per_head_first[head].item() * 40)
    print(f'  head {head:>2d}  {per_head_first[head].item():.4f}  {bar}')

_most = per_head_first.argmax().item()
_least = per_head_first.argmin().item()
print()
show_attention(official.attentions[LOOK_AT], _most,
               f'Layer {LOOK_AT}, head {_most}（首位占比最高）')
print()
show_attention(official.attentions[LOOK_AT], _least,
               f'Layer {LOOK_AT}, head {_least}（首位占比最低）')

Layer 24 各 head 的首位占比：
  head  0  0.8218  ████████████████████████████████
  head  1  0.9889  ███████████████████████████████████████
  head  2  0.5566  ██████████████████████
  head  3  0.9718  ██████████████████████████████████████
  head  4  0.8888  ███████████████████████████████████
  head  5  0.9526  ██████████████████████████████████████
  head  6  0.9389  █████████████████████████████████████
  head  7  0.9547  ██████████████████████████████████████
  head  8  0.9852  ███████████████████████████████████████
  head  9  0.6905  ███████████████████████████
  head 10  0.9660  ██████████████████████████████████████
  head 11  0.9601  ██████████████████████████████████████
  head 12  0.9403  █████████████████████████████████████
  head 13  0.9621  ██████████████████████████████████████
  head 14  0.8179  ████████████████████████████████
  head 15  0.9084  ████████████████████████████████████

Layer 24, head 1（首位占比最高）  (行=query 位置，列=key 位置，值 ×100)
                北京   是中国     的    首都 

## 12. 验证体系汇总

把散落在各节的验证结果集中列出，确认没有遗漏的节点，也确认没有"看起来通过其实没测"的项。

In [58]:
print(f'共 {len(VERIFICATIONS)} 项验证\n')
print(f'{"项目":<38s} {"max_abs":>11s}  结果')
print('─' * 58)
for name, ok, max_abs in VERIFICATIONS:
    print(f'{name:<38s} {max_abs:>11.3e}  {"✓" if ok else "✗ 失败"}')
print('─' * 58)

_failed = [name for name, ok, _ in VERIFICATIONS if not ok]
print(f'通过 {len(VERIFICATIONS) - len(_failed)} / {len(VERIFICATIONS)}')
if _failed:
    print(f'失败项: {_failed}')
else:
    print('全部通过。从 input_ids 到 Top-K 预测的整条链路已逐节点对齐。')

共 38 项验证

项目                                         max_abs  结果
──────────────────────────────────────────────────────────
my_linear vs q_proj                      0.000e+00  ✓
my_rmsnorm vs input_layernorm            0.000e+00  ✓
my_rmsnorm vs post_attn_norm             0.000e+00  ✓
my_silu vs F.silu                        1.192e-07  ✓
my_embedding vs embed_tokens             0.000e+00  ✓
my_embedding vs hidden_states[0]         0.000e+00  ✓
my_position_ids                          0.000e+00  ✓
my_rope cos                              0.000e+00  ✓
my_rope sin                              0.000e+00  ✓
my_inv_freq                              0.000e+00  ✓
my_causal_mask                           0.000e+00  ✓
L0 input_layernorm                       0.000e+00  ✓
L0 q_proj                                0.000e+00  ✓
L0 k_proj                                0.000e+00  ✓
L0 v_proj                                0.000e+00  ✓
L0 q_norm                                0.000e+00  ✓
L0 k_norm   

In [59]:
# 端到端的最终确认
_end_to_end = [
    ('logits 形状一致', tuple(my_logits.shape) == tuple(official.logits.shape)),
    ('argmax 预测全部一致', torch.equal(my_logits[0].argmax(-1),
                                    official.logits[0].argmax(-1))),
    (f'Top-{TOP_K} 排序一致', torch.equal(my_last.topk(TOP_K).indices,
                                      official_last.topk(TOP_K).indices)),
    ('logits 相对误差 < 1e-5',
     (my_logits - official.logits).abs().max().item()
     / official.logits.abs().max().item() < 1e-5),
    ('28 层 hidden state 全部对齐', all(row[5] for row in layer_errors)),
    ('28 层 attention 权重全部对齐', all(row[4] < 1e-5 for row in layer_errors)),
]
for name, ok in _end_to_end:
    print(f'{"✓" if ok else "✗"} {name}')
print()
print(f'端到端闭环成立: {all(ok for _, ok in _end_to_end)}')

✓ logits 形状一致
✓ argmax 预测全部一致
✓ Top-10 排序一致
✓ logits 相对误差 < 1e-5
✓ 28 层 hidden state 全部对齐
✓ 28 层 attention 权重全部对齐

端到端闭环成立: True


## 13. 实验发现 / Experiment Findings

以下内容由实际运行数据产生，不是预先写好的结论。

### 13.1 复现结果

从 `input_ids` 到 Top-K 预测的每一个节点都与官方实现对齐。Layer 0 的 25 个节点里，
`q_proj`、`k_proj`、`v_proj`、`q_norm`、`k_norm`、RoPE、`repeat_kv`、attention 权重、
`o_proj`、`gate_proj`、`up_proj` 全部是 **max_abs = 0**，即逐位相同。
只有经过多次累加的 `down_proj` 及其下游出现 1e-7 量级差异，来自 float32 矩阵乘的求和顺序不同。

28 层各自独立向前推进（每层吃自己上一层的输出，不借用官方中间态），
尺度相对误差从 Layer 0 的 3.9e-08 增长到 Layer 27 的 1.1e-06，增长了约 27 倍，
与 28 次矩阵乘的误差累积量级相符。**9 个位置的 argmax 预测和 Top-10 排序完全一致。**

### 13.2 与预期不同的地方

**（1）`hidden_states` 的下标语义不均匀。**
最初假设 `hidden_states[28]` 是 Layer 27 的输出，验证时那一层报出 460 的误差，
而 attention 权重完全正确。这种"只有最后一层错、且错得离谱"的形态说明是语义理解错误，
不是数值问题。查证后确认 `hidden_states[28]` 已经过 final RMSNorm，
`torch.equal(hidden_states[28], final_norm 的 hook 输出)` 为 True。
这个坑的定位方式印证了 §十一 的做法有效：**逐层验证能立刻指出问题在哪一层，
而不是等到 logits 不对再回头排查。**

**（2）`rope_theta` 在 config 对象里换了位置。**
`config.json` 里它是顶层字段，但 transformers 5.15 把它移进了 `config.rope_parameters` 字典，
`hasattr(config, 'rope_theta')` 返回 False。凭记忆写 `config.rope_theta` 会直接报错。

**（3)`q_norm` / `k_norm` 的位置比预想的更靠里。**
它作用在 **reshape 之后**的 `head_dim=128` 维上，而不是 1024 维上；
`weight` 长度只有 128，被 16 个（或 8 个）头共享；V 完全没有 norm。
源码把 projection、view、norm、transpose 四个操作写在一行里，容易读漏。

### 13.3 层与层之间的差异很大

按 §11 的统计量，28 层明显分成三段：

| 层 | 相对变化 | 行为 |
|---|---|---|
| 0–2 | 12.4 / 0.60 / **156.5** | 剧烈重写。Layer 2 把残差流范数从 41 拉到 6441 |
| 3–25 | 0.003 – 0.052 | 每层只做百分之几的改动 |
| 26–27 | 0.090 / **0.904** | Layer 27 把范数从 6635 压回 1510 |

中间 23 层的相对改动中位数只有约 0.8%，但这不等于它们没用——残差流被 Layer 2 写入的
巨大常量支配，分母很大，所以相对值被压低了。

### 13.4 两个数量级异常的激活通道

验证过程中撞见的现象，不在原计划的观察清单里：

- **位置 0，通道 35**：在 Layer 2 内部被一次性写入约 **6392**，之后 23 层几乎原样保留，
  final norm 之后变成 −2.1 被彻底压掉。同位置其他通道的中位量级只有 1.5，相差三个数量级。
- **位置 2，通道 27**：从中段开始逐层累积，Layer 26 输出时约 **99**，
  且在 final norm 之后仍是全张量最大的元素（78.9）。

第一个通道直接解释了 §9.2 那个"从 Layer 2 起恒定不变的 max_abs = 1.465e-03"：
float32 在 6392 附近的最小间隔是 7.6e-04，观察到的误差只有约 2 个 ULP。
**这也说明用固定绝对公差判定这种张量是错的**，必须看尺度相对误差。

### 13.5 注意力大量集中在位置 0

"首位占比"从 Layer 3 起长期维持在 0.55–0.89。Layer 24 的 16 个 head 里，
有 11 个把 90% 以上的权重压在位置 0 上，head 1 达到 98.9%。
这与 §13.4 的通道 35 出现在位置 0 是同一现象的两面。

相比之下 Layer 0 的形态完全不同：对角占比 0.63（倾向于看自己），首位占比只有 0.06。

### 13.6 结构上确认的几件事

- **`lm_head` 与 `embed_tokens` 共享同一块内存**（`data_ptr()` 相同），
  这张 `[151936, 1024]` 的表占模型总参数的 26.1%，且不存在于 `model.safetensors` 里。
- **28 层共用同一个 `attention_mask` 对象和同一组 `(cos, sin)`**，
  RoPE 在模型级别算一次，不在 Attention 内部重复计算。
- **`q_proj` 把 1024 升到 2048**，比 hidden_size 更宽。`head_dim=128` 是配置写死的，
  不是 `hidden_size / num_heads`。
- **RoPE 是保长变换**：向量模长在旋转前后相对变化为 0；位置 0 的 cos=1、sin=0，
  该位置的 q/k 完全不变。
- **`hidden_states` 全部 29 个张量形状相同**，都是 `[1, 9, 1024]`。

## 14. 小结

本实验走通的完整链路：

```text
自然语言
   ↓  tokenizer（未解剖）
input_ids                 [1, 9]
   ↓  embed_tokens = 查表
hidden                    [1, 9, 1024]
   ↓  ┌─ 28 × Decoder Layer ────────────────────────┐
   ↓  │ RMSNorm → q/k/v_proj → q/k_norm → RoPE      │
   ↓  │ → repeat_kv → QKᵀ·scaling → +mask → softmax │
   ↓  │ → ×V → o_proj → ⊕residual                   │
   ↓  │ → RMSNorm → SiLU(gate)*up → down → ⊕residual│
   ↓  └─────────────────────────────────────────────┘
hidden                    [1, 9, 1024]     ← 保形
   ↓  final RMSNorm
   ↓  lm_head（复用 embedding 那张表）
logits                    [1, 9, 151936]
   ↓  取最后位置 → Top-K → decode
'首都'  (95.94%)
```

每一步都用 PyTorch 基础算子重写并与官方实现对齐，误差全部在 float32 精度范围内，
最终预测完全一致。

完成本实验后，应该能回答：

1. 一个 token 从 `input_ids` 到 logits，中间经过哪些具体的矩阵运算？
2. Q、K、V 三个投影的输出维度为什么不同，GQA 在哪一步把它们对齐？
3. `q_norm` 归一化的是哪一维，为什么是 128 而不是 1024 或 2048？
4. RoPE 的 cos/sin 在哪里计算，为什么 28 层可以共用？
5. causal mask 是怎么起作用的，为什么是加法而不是乘法？
6. 残差连接加的是归一化前还是归一化后的张量？
7. 为什么 `hidden_states` 有 29 个而不是 28 个，最后一个是什么？
8. 为什么 `model.safetensors` 里找不到 `lm_head.weight`？